[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab04_medical_image_classification.ipynb)

# Lab 4: การเรียนรู้เชิงลึก (Deep Learning) สำหรับการจำแนกภาพทางการแพทย์

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันพฤหัสบดีที่ 8 ตุลาคม 2026 · **เวลาโดยประมาณ:** 90 นาที

## วัตถุประสงค์การเรียนรู้

- สร้างและฝึกโครงข่ายประสาทเทียมแบบคอนโวลูชัน (convolutional neural network: CNN) ด้วยการเรียนรู้แบบถ่ายโอน (transfer learning) กับภาพถ่ายรังสีทรวงอก
- ใช้ชุดข้อมูล (dataset) สาธารณะ MedMNIST ซึ่งไม่ต้องดาวน์โหลดด้วยตนเองและใช้งานบน Colab ได้ทันที
- ประเมินผลด้วยเมทริกซ์ความสับสน (confusion matrix) ความไว (sensitivity) และความจำเพาะ (specificity) รวมทั้งพิจารณาภาพที่โมเดลจำแนกผิด
- เข้าใจภาพรวมของเครื่องมือกำกับข้อมูลภาพ (image labeling) เช่น LabelMe และ CVAT และการแบ่งส่วนภาพ (segmentation)

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง (คลิกปุ่ม ▶ ด้านซ้ายของเซลล์ หรือกด **Shift+Enter**) แบบฝึกหัด ⭐ **Exercise** แต่ละข้อมีเซลล์ ✅ **เฉลย** อยู่ถัดไป ผู้เรียนควรทดลองทำด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำขั้นตอนถัดไป จากนั้นให้ตอบคำถาม ✍️ ในเซลล์ข้อความ ซึ่งถือเป็นชิ้นงานที่ต้องส่ง
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้** อยู่ท้ายโน้ตบุ๊ก (notebook) และไม่นับเป็นส่วนของชิ้นงานที่ต้องส่ง
> - หากพบปัญหาหรือข้อผิดพลาด (error) สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom

> **⚡ การเปิดใช้ GPU (โปรดดำเนินการก่อนรันเซลล์ใด ๆ)**
> การฝึกโครงข่ายประสาทเทียมบน GPU (หน่วยประมวลผลกราฟิก ซึ่งคำนวณแบบขนานได้จำนวนมาก) ใช้เวลาน้อยกว่าบน CPU อย่างมาก
> ใน Colab ให้เลือก **Runtime → Change runtime type → Hardware accelerator: T4 GPU → Save**
> การเปลี่ยน runtime จะเริ่มโน้ตบุ๊กใหม่ จึงควรดำเนินการ *ก่อน* รันเซลล์แรก (หรือรันใหม่ตั้งแต่ต้นหลังเปลี่ยน)
> หาก Colab แจ้งว่าไม่มี GPU ให้ใช้งาน ปฏิบัติการนี้ยังสามารถรันบน CPU ได้ แต่การฝึกโมเดลจะใช้เวลานานขึ้นหลายนาที

ปฏิบัติการทั้งหมดรันได้บน [Google Colab](https://colab.research.google.com) รุ่นที่ไม่มีค่าใช้จ่าย ตารางอันดับ (leaderboard) 🏆 ของรายวิชาอยู่ที่ [Hugging Face](https://huggingface.co/spaces/wdittaya/aidh-leaderboard) ผู้เรียนจำเป็นต้องระบุ `NAME` และ `JOIN_CODE` ในเซลล์การตั้งค่าเฉพาะเมื่อต้องการเข้าร่วมโจทย์ท้าทายเท่านั้น

In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เป็นการ "เตรียมความพร้อม" ก่อนเริ่มปฏิบัติการ โดยทำ 3 อย่าง ได้แก่
#   1) ติดตั้งเครื่องมือ (ไลบรารี) เพิ่มเติมที่จำเป็นต้องใช้ใน Google Colab
#   2) กำหนดค่าพื้นฐาน เช่น รหัสปฏิบัติการ ชื่อที่จะแสดง และรหัสเข้าร่วมตารางอันดับ
#   3) สร้าง "ฟังก์ชัน" (ชุดคำสั่งสำเร็จรูปที่เรียกใช้ซ้ำได้) 2 ตัว คือ
#        - lab_data(...)              : ดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#        - submit_to_leaderboard(...) : ส่งผลการทำนายไปยังตารางอันดับ (ไม่บังคับ)
#   ผู้เรียนเพียงกดรันเซลล์นี้ 1 ครั้ง ไม่จำเป็นต้องแก้ไขโค้ด
#   (ยกเว้นต้องการเข้าร่วมตารางอันดับ ให้กรอก NAME และ JOIN_CODE ด้านล่าง)
# ============================================================================

# ติดตั้งไลบรารีเพิ่มเติมลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - gradio_client   : ไลบรารีสำหรับติดต่อกับเว็บแอปตารางอันดับ (Hugging Face Space)
#   - huggingface_hub : ไลบรารีสำหรับดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face
!pip -q install gradio_client huggingface_hub

# สร้างตัวแปร (ช่องเก็บค่า) HF_OWNER แล้วใส่ข้อความ "wdittaya" ไว้
#   เครื่องหมาย = หมายถึง "นำค่าทางขวา ไปเก็บไว้ในชื่อทางซ้าย"
#   ข้อความในภาษา Python ต้องอยู่ในเครื่องหมายคำพูด " "
HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
# รหัสของปฏิบัติการนี้ ใช้ระบุว่าผลที่ส่งเป็นของปฏิบัติการใด (ไม่ต้องแก้ไข)
LAB_ID    = "lab04"
# ชื่อที่จะแสดงบนตารางอันดับ: พิมพ์ชื่อไว้ระหว่างเครื่องหมาย " " เช่น NAME = "หมอเอ"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
# รหัสเข้าร่วมตารางอันดับ: พิมพ์รหัสที่ผู้สอนแจ้งไว้ระหว่างเครื่องหมาย " "
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

# สร้างชื่อเว็บแอปตารางอันดับ โดยนำ HF_OWNER มาต่อกับข้อความ
#   - f"..." (f-string) คือข้อความที่แทรกค่าตัวแปรได้ โดยเขียนชื่อตัวแปรไว้ใน { }
#   - ผลลัพธ์คือ "wdittaya/aidh-leaderboard"
SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
# สร้างชื่อคลังข้อมูล (dataset) ของรายวิชาในลักษณะเดียวกัน ผลลัพธ์คือ "wdittaya/aidh-lab-data"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

# นำเข้า (import) ฟังก์ชัน hf_hub_download จากไลบรารี huggingface_hub
#   ฟังก์ชันนี้ใช้ดาวน์โหลดไฟล์จากเว็บไซต์ Hugging Face มาเก็บไว้ในเครื่อง
from huggingface_hub import hf_hub_download

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน lab_data สำหรับดาวน์โหลดไฟล์ข้อมูลของรายวิชา
#   - def คือคำสั่งสร้างฟังก์ชันใหม่ ตามด้วยชื่อฟังก์ชันและพารามิเตอร์ในวงเล็บ
#   - path : พารามิเตอร์ (ค่าที่ผู้ใช้ส่งเข้ามา) คือชื่อไฟล์ที่ต้องการ เช่น "pima/test_ids.csv"
#   - บรรทัดที่ย่อหน้าเข้าไปด้านล่าง คือคำสั่งที่อยู่ภายในฟังก์ชัน
# ----------------------------------------------------------------------------
def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    # เรียก hf_hub_download(...) เพื่อดาวน์โหลดไฟล์ แล้วส่งตำแหน่งไฟล์ในเครื่องกลับไป (return)
    #   - DATA_REPO           : ชื่อคลังข้อมูลบน Hugging Face ที่เก็บไฟล์
    #   - path                : ชื่อไฟล์ที่ต้องการดาวน์โหลดภายในคลังข้อมูลนั้น
    #   - repo_type="dataset" : ระบุว่าคลังนี้เป็นประเภท "ชุดข้อมูล" (ไม่ใช่โมเดล)
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

# ----------------------------------------------------------------------------
# สร้างฟังก์ชัน submit_to_leaderboard สำหรับส่งไฟล์ผลการทำนายไปยังตารางอันดับ
#   - path   : ชื่อไฟล์ผลการทำนายที่ต้องการส่ง
#   - lab=LAB_ID : รหัสปฏิบัติการ หากไม่ระบุ จะใช้ค่า LAB_ID ที่กำหนดไว้ด้านบนโดยอัตโนมัติ
# ----------------------------------------------------------------------------
def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    # นำเข้าเครื่องมือ 2 ตัวจากไลบรารี gradio_client
    #   - Client      : ใช้เชื่อมต่อกับเว็บแอปตารางอันดับ
    #   - handle_file : ใช้เตรียมไฟล์ในเครื่องให้อยู่ในรูปแบบที่ส่งขึ้นเว็บแอปได้
    from gradio_client import Client, handle_file
    # assert คือการตรวจสอบเงื่อนไข หากเงื่อนไขเป็นเท็จ โปรแกรมจะหยุดและแสดงข้อความเตือน
    #   - NAME and JOIN_CODE : เงื่อนไขว่าต้องกรอกทั้ง NAME และ JOIN_CODE แล้ว (ไม่เป็นข้อความว่าง)
    #   - ข้อความหลังเครื่องหมายจุลภาค คือข้อความเตือนที่จะแสดงเมื่อยังไม่ได้กรอก
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    # เชื่อมต่อกับเว็บแอปตารางอันดับ แล้วเก็บการเชื่อมต่อไว้ในตัวแปร client
    #   - SPACE_ID      : ชื่อเว็บแอปตารางอันดับที่กำหนดไว้ด้านบน
    #   - verbose=False : ไม่ต้องแสดงข้อความรายละเอียดระหว่างเชื่อมต่อ
    client = Client(SPACE_ID, verbose=False)
    # ส่งข้อมูลไปยังเว็บแอปด้วย client.predict(...) แล้วแสดงผลลัพธ์ (คะแนนและอันดับ) ด้วย print(...)
    #   - lab=lab                 : รหัสปฏิบัติการ
    #   - name=NAME               : ชื่อที่จะแสดงบนตารางอันดับ
    #   - code=JOIN_CODE          : รหัสเข้าร่วม
    #   - file=handle_file(path)  : ไฟล์ผลการทำนายที่เตรียมพร้อมสำหรับส่ง
    #   - api_name="/submit"      : ชื่อช่องทางรับข้อมูลของเว็บแอป (ส่วน "ส่งผล")
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

# แสดงข้อความยืนยันว่าการตั้งค่าเสร็จแล้ว ด้วยฟังก์ชัน print(...) (พิมพ์ข้อความออกทางหน้าจอ)
#   - ข้อความเป็น f-string จึงแทรกค่า SPACE_ID, LAB_ID และ NAME ลงในข้อความได้
#   - NAME or '(ยังไม่ได้ระบุ ...)' : หาก NAME ว่าง จะแสดงข้อความในวงเล็บแทน
print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ส่วน A — ชุดข้อมูล PneumoniaMNIST (MedMNIST)

ภาพถ่ายรังสีทรวงอกของผู้ป่วยเด็กซึ่งกำกับว่า **ปกติ (normal, 0)** หรือ **ปอดอักเสบ (pneumonia, 1)** โดยย่อขนาดเหลือ 28×28 พิกเซล ชุดข้อมูลนี้มีขนาดเล็ก อยู่ในรูปแบบมาตรฐาน และติดตั้งได้ด้วย `pip` จึงเหมาะกับ GPU ของ Colab รุ่นที่ไม่มีค่าใช้จ่าย

เซลล์ถัดไปติดตั้งไลบรารีและแสดงอุปกรณ์ที่ใช้ประมวลผล ได้แก่ GPU (`cuda`) หรือ CPU

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   เซลล์นี้เตรียม "เครื่องมือ" ทั้งหมดที่ใช้ในปฏิบัติการการจำแนกภาพถ่ายรังสีทรวงอก ได้แก่
#   1) ติดตั้งไลบรารี medmnist ซึ่งเป็นแหล่งชุดข้อมูลภาพทางการแพทย์ขนาดเล็ก (MedMNIST)
#   2) นำเข้า (import) ไลบรารีหลัก ได้แก่
#        - PyTorch (torch)  : ไลบรารีสำหรับสร้างและฝึกโครงข่ายประสาทเทียม (neural network)
#        - NumPy (np)       : ไลบรารีสำหรับคำนวณตัวเลขเป็นชุด (array)
#        - Matplotlib (plt) : ไลบรารีสำหรับวาดกราฟและแสดงภาพ
#   3) ตั้งค่าการสุ่มให้ได้ผลซ้ำเดิม และเลือก "อุปกรณ์ประมวลผล" ว่าจะใช้ GPU หรือ CPU
#   สิ่งที่ผู้เรียนจะเห็น: ข้อความแจ้งว่ากำลังใช้ GPU (cuda) หรือ CPU
#   (GPU คือหน่วยประมวลผลกราฟิก ช่วยให้ฝึกโมเดลภาพได้เร็วกว่า CPU หลายเท่า)
# ============================================================================

# ติดตั้งไลบรารี medmnist ลงในเครื่อง Colab
#   - เครื่องหมาย ! ด้านหน้า หมายถึงสั่งงานระบบปฏิบัติการ (ไม่ใช่คำสั่งภาษา Python)
#   - pip install : คำสั่งติดตั้งไลบรารีของ Python
#   - -q (quiet)  : ให้แสดงข้อความระหว่างติดตั้งให้น้อยที่สุด
#   - medmnist    : ไลบรารีที่รวบรวมชุดข้อมูลภาพทางการแพทย์ เช่น PneumoniaMNIST (ภาพถ่ายรังสีทรวงอกเด็ก)
!pip -q install medmnist
# นำเข้า (import) ไลบรารี torch (PyTorch) และโมดูลย่อย torch.nn
#   - import ... คือคำสั่งนำไลบรารีเข้ามาให้ใช้งานในโน้ตบุ๊ก
#   - torch            : ไลบรารีหลักของ PyTorch ใช้คำนวณกับ "เทนเซอร์" (tensor = ชุดตัวเลขหลายมิติ เช่น ภาพ)
#   - torch.nn as nn   : โมดูลย่อยสำหรับสร้างชั้น (layer) ต่าง ๆ ของโครงข่ายประสาทเทียม
#                        "as nn" คือการตั้งชื่อย่อ เพื่อพิมพ์ nn แทน torch.nn ได้
#   - เครื่องหมายจุลภาค , ช่วยให้นำเข้าหลายรายการในบรรทัดเดียว
import torch, torch.nn as nn
# นำเข้าไลบรารีสำหรับคำนวณและวาดกราฟ
#   - numpy as np               : ไลบรารีคำนวณตัวเลขเป็นชุด (array) ตั้งชื่อย่อว่า np
#   - matplotlib.pyplot as plt  : ส่วนวาดกราฟ/แสดงภาพของไลบรารี matplotlib ตั้งชื่อย่อว่า plt
import numpy as np, matplotlib.pyplot as plt
# นำเข้า 2 สิ่งจากไลบรารี medmnist
#   - from ... import ... : นำเข้าเฉพาะบางส่วนของไลบรารี
#   - PneumoniaMNIST : "คลาส" (แม่แบบ) สำหรับโหลดชุดข้อมูลภาพถ่ายรังสีทรวงอก ปกติ / ปอดอักเสบ
#   - INFO           : พจนานุกรม (dict) ที่เก็บรายละเอียดของชุดข้อมูลแต่ละชุด เช่น ความหมายของป้ายกำกับ
from medmnist import PneumoniaMNIST, INFO

# กำหนด "เมล็ดพันธุ์การสุ่ม" (random seed) ของ PyTorch ให้เป็น 0
#   - torch.manual_seed(...) : ฟังก์ชันของ PyTorch ที่ทำให้ตัวเลขสุ่มออกมาเหมือนเดิมทุกครั้งที่รัน
#   - 0 : ค่าเมล็ดพันธุ์ (เลขใดก็ได้ ขอเพียงใช้เลขเดิม) ช่วยให้ผลลัพธ์ใกล้เคียงกันระหว่างผู้เรียน
#   (หมายเหตุ: บน GPU ผลอาจยังต่างกันเล็กน้อย เนื่องจากการคำนวณบางส่วนมีความไม่แน่นอน)
torch.manual_seed(0)
# เลือกอุปกรณ์ประมวลผล แล้วเก็บไว้ในตัวแปร device
#   - torch.cuda.is_available() : ฟังก์ชันของ PyTorch ตรวจว่าเครื่องนี้มี GPU (cuda) ให้ใช้หรือไม่ ได้ผลเป็น True/False
#   - รูปแบบ  ค่า_A if เงื่อนไข else ค่า_B  คือ "ถ้าเงื่อนไขเป็นจริงใช้ค่า_A มิฉะนั้นใช้ค่า_B"
#   - ผลลัพธ์: device = "cuda" หากมี GPU หรือ "cpu" หากไม่มี
device = "cuda" if torch.cuda.is_available() else "cpu"
# แสดงข้อความบอกอุปกรณ์ที่ใช้ด้วย print(...) (ฟังก์ชันพื้นฐานของ Python สำหรับพิมพ์ข้อความออกหน้าจอ)
#   - "อุปกรณ์:" : ข้อความนำหน้า
#   - device     : ค่าของตัวแปร device ("cuda" หรือ "cpu")
#   - ข้อความสุดท้ายเลือกด้วย if/else: หาก device == "cuda" (== คือ "เท่ากับหรือไม่") แสดง "(GPU)"
#     มิฉะนั้นแสดงคำเตือนว่ากำลังใช้ CPU ซึ่งช้ากว่า
#   - print จะเว้นวรรคคั่นระหว่างแต่ละค่าที่คั่นด้วยจุลภาคให้อัตโนมัติ
print("อุปกรณ์:", device, "(GPU)" if device == "cuda" else "(CPU - ใช้งานได้แต่ช้ากว่า โปรดดูวิธีเปิดใช้ GPU ด้านบน)")

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ดาวน์โหลดชุดข้อมูล PneumoniaMNIST (ภาพถ่ายรังสีทรวงอกเด็ก ขนาด 28×28 พิกเซล) แบ่งเป็น 3 ชุด
#     - train (ชุดฝึก)          : ใช้สอนโมเดล
#     - val   (ชุดตรวจสอบ, validation) : ใช้ตรวจสอบผลระหว่างฝึก
#     - test  (ชุดทดสอบ)        : แยกไว้ประเมินผลขั้นสุดท้ายเท่านั้น (624 ภาพ)
#   พร้อมกำหนดวิธี "แปลงภาพ" (transform) ให้เป็นตัวเลขที่โครงข่ายประสาทเทียมใช้ได้
#   สิ่งที่ผู้เรียนจะเห็น: ความหมายของป้ายกำกับ (0 = ปกติ, 1 = ปอดอักเสบ) จำนวนภาพในแต่ละชุด
#   และสัดส่วนภาพปอดอักเสบ ซึ่งแสดงให้เห็น "ความไม่สมดุลของคลาส" (class imbalance)
# ============================================================================

# ดาวน์โหลดข้อมูล 3 ชุด: train (สำหรับฝึกโมเดล), val (สำหรับตรวจสอบระหว่างฝึก), test (สำหรับประเมินผลขั้นสุดท้าย)
# นำเข้าโมดูล transforms จากไลบรารี torchvision (ส่วนเสริมของ PyTorch สำหรับงานภาพ)
#   transforms รวบรวมเครื่องมือแปลงภาพ เช่น แปลงเป็นตัวเลข ปรับช่วงค่า หมุนภาพ
from torchvision import transforms
# ----------------------------------------------------------------------------
# สร้างขั้นตอนการแปลงภาพ แล้วเก็บไว้ในตัวแปร tf
#   - transforms.Compose([...]) : นำการแปลงหลายขั้นมาต่อกันเป็นลำดับ ภาพจะผ่านทีละขั้นจากบนลงล่าง
#   - [ ... ] คือ list (รายการ) ของขั้นตอนการแปลง
# ----------------------------------------------------------------------------
tf = transforms.Compose([
    # ขั้นที่ 1: transforms.ToTensor() แปลงภาพเป็นเทนเซอร์ (ชุดตัวเลข) และหารด้วย 255
    #   ทำให้ค่าความสว่างของพิกเซลจาก 0-255 กลายเป็น 0-1
    transforms.ToTensor(),              # ค่าพิกเซล 0-255 -> ตัวเลข 0-1
    # ขั้นที่ 2: transforms.Normalize(mean, std) ปรับมาตรฐานค่า โดยคำนวณ (ค่า - mean) / std
    #   - [.5] (ค่าแรก)  : ค่าเฉลี่ย (mean) = 0.5 ของช่องสีเดียว (ภาพระดับเทามี 1 ช่อง จึงมีค่าเดียวใน list)
    #   - [.5] (ค่าที่สอง): ส่วนเบี่ยงเบนมาตรฐาน (std) = 0.5
    #   ผลคือ 0 -> -1 และ 1 -> 1 จึงได้ช่วงค่า -1 ถึง 1
    transforms.Normalize([.5], [.5]),   # -> ตัวเลข -1 ถึง 1 (ช่วงค่าที่เหมาะกับโครงข่าย)
])
# โหลดชุดฝึก (train) แล้วเก็บไว้ในตัวแปร train_ds (ds ย่อมาจาก dataset)
#   - PneumoniaMNIST(...) : คลาสจากไลบรารี medmnist สำหรับโหลดชุดข้อมูลนี้
#   - split="train"  : เลือกส่วน "ชุดฝึก"
#   - transform=tf   : ให้แปลงทุกภาพด้วยขั้นตอน tf ที่สร้างไว้ด้านบน ทุกครั้งที่ดึงภาพออกมา
#   - download=True  : หากยังไม่มีไฟล์ในเครื่อง ให้ดาวน์โหลดจากอินเทอร์เน็ตอัตโนมัติ
#   - size=28        : ใช้ภาพขนาด 28×28 พิกเซล (ขนาดเล็กที่สุด ประมวลผลเร็ว)
train_ds = PneumoniaMNIST(split="train", transform=tf, download=True, size=28)
# โหลดชุดตรวจสอบ (validation) ด้วยพารามิเตอร์ชุดเดียวกัน ยกเว้น split="val"
val_ds   = PneumoniaMNIST(split="val",   transform=tf, download=True, size=28)
# โหลดชุดทดสอบ (test) ด้วยพารามิเตอร์ชุดเดียวกัน ยกเว้น split="test"
test_ds  = PneumoniaMNIST(split="test",  transform=tf, download=True, size=28)
# แสดงความหมายของป้ายกำกับ (label) ของชุดข้อมูลนี้
#   - INFO["pneumoniamnist"] : ดึงข้อมูลรายละเอียดของชุด pneumoniamnist จากพจนานุกรม INFO
#     (การใช้ [ "ชื่อ" ] คือการ "ดึงค่าตามชื่อกุญแจ (key)" จากพจนานุกรม dict)
#   - ["label"] : ดึงต่อเฉพาะส่วนความหมายของป้ายกำกับ ผลคือ {'0': 'normal', '1': 'pneumonia'}
print(INFO["pneumoniamnist"]["label"])
# แสดงจำนวนภาพของแต่ละชุด
#   - len(...) : ฟังก์ชันพื้นฐานของ Python นับจำนวนสมาชิก (ในที่นี้คือจำนวนภาพ)
print("train/val/test:", len(train_ds), len(val_ds), len(test_ds))
# ----------------------------------------------------------------------------
# วนลูปนับจำนวนภาพปกติ/ปอดอักเสบในชุดฝึกและชุดทดสอบ
#   - for ... in ... : คำสั่งวนซ้ำ ทำคำสั่งที่ย่อหน้าด้านล่างกับสมาชิกทีละตัวใน list
#   - [("train", train_ds), ("test", test_ds)] : list ของคู่ค่า (tuple) ชื่อชุด กับ ชุดข้อมูล
#   - name, ds : ในแต่ละรอบ แยกคู่ค่าออก ให้ name = ชื่อชุด และ ds = ชุดข้อมูล
# ----------------------------------------------------------------------------
for name, ds in [("train", train_ds), ("test", test_ds)]:
    # นับจำนวนภาพปอดอักเสบ เก็บไว้ใน n_pneu
    #   - ds.labels : ชุดป้ายกำกับของทุกภาพ (0 = ปกติ, 1 = ปอดอักเสบ)
    #   - .sum()    : รวมค่าทั้งหมด เนื่องจากปอดอักเสบ = 1 ผลรวมจึงเท่ากับจำนวนภาพปอดอักเสบ
    #   - int(...)  : แปลงผลให้เป็นจำนวนเต็มของ Python
    n_pneu = int(ds.labels.sum())
    # แสดงผลสรุปด้วย f-string (ข้อความที่แทรกค่าตัวแปรใน { } ได้)
    #   - {name:5s}            : แสดงชื่อชุดเป็นข้อความ (s) กว้าง 5 ตัวอักษร เพื่อให้จัดแนวตรงกัน
    #   - {len(ds)-n_pneu}     : จำนวนภาพปกติ = จำนวนภาพทั้งหมด - จำนวนภาพปอดอักเสบ
    #   - {n_pneu}             : จำนวนภาพปอดอักเสบ
    #   - {n_pneu/len(ds):.0%} : สัดส่วนปอดอักเสบ แสดงเป็นร้อยละ (%) ไม่มีทศนิยม (.0)
    print(f"{name:5s}: ปกติ {len(ds)-n_pneu} ภาพ, ปอดอักเสบ {n_pneu} ภาพ (ปอดอักเสบ {n_pneu/len(ds):.0%})")

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   แสดงตัวอย่างภาพถ่ายรังสีทรวงอก 8 ภาพแรกจากชุดฝึก พร้อมป้ายกำกับ (normal / pneumonia)
#   เพื่อให้ผู้เรียนเห็นลักษณะข้อมูลจริงที่โมเดลจะเรียนรู้ ก่อนเริ่มฝึก
#   สิ่งที่ผู้เรียนจะเห็น: ภาพระดับเทาขนาดเล็ก 8 ภาพเรียงในแถวเดียว
#   (สังเกตว่าภาพขนาด 28×28 พิกเซลมีรายละเอียดต่ำมากเมื่อเทียบกับภาพถ่ายรังสีจริง)
# ============================================================================

# แสดงตัวอย่างภาพ (ข้อความบนกราฟเป็นภาษาอังกฤษ เนื่องจากฟอนต์เริ่มต้นของกราฟไม่รองรับภาษาไทย)
# สร้างพื้นที่วาดภาพ (figure) ที่แบ่งเป็นช่องย่อย ด้วย plt.subplots(...) จากไลบรารี matplotlib
#   - 1, 8           : จำนวนแถว 1 แถว และจำนวนคอลัมน์ 8 คอลัมน์ (รวม 8 ช่องภาพ)
#   - figsize=(16, 2.6) : ขนาดของรูปทั้งหมด กว้าง 16 นิ้ว สูง 2.6 นิ้ว
#   - ผลลัพธ์ 2 ค่า: fig = รูปทั้งหมด, axes = ชุดของช่องภาพย่อย (axes) ทั้ง 8 ช่อง
fig, axes = plt.subplots(1, 8, figsize=(16, 2.6))
# วนลูปจับคู่ "ช่องภาพ" กับ "ภาพจากชุดฝึก" ทีละคู่
#   - zip(axes, train_ds) : ฟังก์ชันพื้นฐานของ Python จับคู่สมาชิกลำดับเดียวกันของ 2 ชุด
#                           และหยุดเมื่อชุดที่สั้นกว่าหมด (axes มี 8 ช่อง จึงได้ 8 ภาพแรก)
#   - ax           : ช่องภาพในรอบนั้น
#   - (img, label) : ข้อมูลแต่ละรายการในชุดข้อมูลประกอบด้วย ภาพ (img) และป้ายกำกับ (label)
for ax, (img, label) in zip(axes, train_ds):
    # แสดงภาพในช่อง ax ด้วย ax.imshow(...)
    #   - img.squeeze() : ตัดมิติที่มีขนาด 1 ออก (ภาพมีรูปร่าง 1×28×28 -> 28×28) เพื่อให้แสดงเป็นภาพ 2 มิติได้
    #   - cmap="gray"   : ใช้โทนสีเทา (ดำ-ขาว) เหมือนฟิล์มเอกซเรย์
    ax.imshow(img.squeeze(), cmap="gray")
    # ตั้งชื่อหัวภาพด้วย ax.set_title(...)
    #   - label[0]   : ป้ายกำกับเก็บเป็นชุดที่มีสมาชิก 1 ตัว จึงดึงตัวแรกด้วย [0] (Python เริ่มนับจาก 0)
    #   - int(...)   : แปลงเป็นจำนวนเต็ม
    #   - "pneumonia" if ... == 1 else "normal" : หากป้ายกำกับเป็น 1 แสดง "pneumonia" มิฉะนั้นแสดง "normal"
    #   - fontsize=9 : ขนาดตัวอักษร 9
    ax.set_title("pneumonia" if int(label[0]) == 1 else "normal", fontsize=9)
    # ซ่อนแกนและตัวเลขบอกตำแหน่งรอบภาพ เพื่อให้ดูภาพได้ชัดเจน
    ax.axis("off")
# สั่งแสดงรูปทั้งหมดออกมาบนหน้าจอ
plt.show()

💡 **คำอธิบายผลลัพธ์** ชุดข้อมูลประกอบด้วยภาพถ่ายรังสีที่มีการกำกับข้อมูลแล้วประมาณ 5,900 ภาพ แบ่งเป็น *train* (ใช้ฝึกโมเดล) *validation* (ใช้ตรวจสอบระหว่างฝึก) และ *test* (624 ภาพ แยกไว้สำหรับประเมินผลขั้นสุดท้าย) ภาพในชุดฝึกประมาณสามในสี่เป็นภาพปอดอักเสบ กล่าวคือ ข้อมูลมี **ความไม่สมดุลของคลาส (class imbalance)** โมเดลที่ตอบว่า "ปอดอักเสบ" ทุกภาพจึงถูกต้องประมาณร้อยละ 75 ของชุดฝึก นอกจากนี้ ภาพขนาด 28×28 พิกเซลมีความละเอียดต่ำกว่าภาพถ่ายรังสีจริงมาก ชุดข้อมูลนี้จึงเหมาะสำหรับการเรียนการสอน ไม่ใช่การวินิจฉัย

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   สร้าง "ตัวป้อนข้อมูล" (DataLoader) สำหรับชุดฝึก ชุดตรวจสอบ และชุดทดสอบ
#   ตัวป้อนข้อมูลจะแบ่งภาพออกเป็นชุดย่อย (batch) ครั้งละหลายภาพ แล้วส่งให้โครงข่ายประมวลผลทีละชุด
#   ซึ่งเร็วกว่าการป้อนทีละภาพ และประหยัดหน่วยความจำกว่าการป้อนทั้งหมดในครั้งเดียว
#   สิ่งที่ได้: ตัวแปร train_loader, val_loader, test_loader (เซลล์นี้ไม่แสดงผลลัพธ์บนหน้าจอ)
# ============================================================================

# ป้อนภาพให้โครงข่ายเป็นชุดย่อย (batch)
# นำเข้าคลาส DataLoader จากโมดูล torch.utils.data ของ PyTorch
from torch.utils.data import DataLoader
# สร้างตัวป้อนข้อมูลสำหรับชุดฝึก
#   - train_ds      : ชุดข้อมูลที่จะป้อน (ชุดฝึก)
#   - batch_size=64 : ป้อนครั้งละ 64 ภาพ
#   - shuffle=True  : สลับลำดับภาพแบบสุ่มทุกรอบการฝึก เพื่อไม่ให้โมเดลจดจำลำดับของภาพ
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
# สร้างตัวป้อนข้อมูลสำหรับชุดตรวจสอบ
#   - val_ds         : ชุดตรวจสอบ
#   - batch_size=128 : ครั้งละ 128 ภาพ (ประเมินผลอย่างเดียวจึงใช้ชุดใหญ่ขึ้นได้)
#   - ไม่ระบุ shuffle จึงใช้ค่าเริ่มต้น shuffle=False (คงลำดับเดิม)
val_loader   = DataLoader(val_ds, batch_size=128)
# สร้างตัวป้อนข้อมูลสำหรับชุดทดสอบ ครั้งละ 128 ภาพ โดยไม่สลับลำดับ
#   การคงลำดับสำคัญมาก เพราะผลการทำนายจะเรียงตรงกับลำดับภาพที่ 0 ถึง 623 (ใช้ส่งตารางอันดับได้)
test_loader  = DataLoader(test_ds, batch_size=128)   # shuffle=False -> คงลำดับ 0..623

## การเรียนรู้แบบถ่ายโอน (transfer learning) ด้วย ResNet-18

**ResNet-18** เป็น CNN ที่ได้รับการฝึกมาแล้วด้วยภาพถ่ายทั่วไปประมาณ 1 ล้านภาพ (ชุดข้อมูล *ImageNet*) จึงสามารถตรวจจับขอบ พื้นผิว และรูปทรงได้ **การเรียนรู้แบบถ่ายโอน** คือการนำความรู้ดังกล่าวมาใช้ต่อ และปรับโครงข่ายเพียงบางส่วนให้เหมาะกับงานใหม่ วิธีนี้ทำให้ได้โมเดลที่มีประสิทธิภาพแม้มีข้อมูลจำนวนจำกัด จึงเป็นแนวทางมาตรฐานในงานภาพทางการแพทย์ ซึ่งภาพที่กำกับข้อมูลแล้วมีต้นทุนสูง

การปรับโครงข่ายมี 2 ส่วน ได้แก่
1. **ชั้นนำเข้า (input layer)** — ภาพ ImageNet เป็นภาพสี (3 ช่องสี (channel): แดง เขียว น้ำเงิน) ขณะที่ภาพถ่ายรังสีเป็นภาพระดับเทา (grayscale) ซึ่งมีเพียง **1** ช่อง
2. **ชั้นส่งออก (output layer)** — ImageNet มี 1,000 คลาส แต่งานนี้มีเพียง **2** คลาส (ปกติ และปอดอักเสบ)

**⭐ แบบฝึกหัดที่ 1 (Exercise 1):** ให้แทนที่ `...` ทั้งสองตำแหน่งในเซลล์ด้านล่างด้วยตัวเลขที่ถูกต้อง
*คำใบ้:* ภาพถ่ายรังสีระดับเทามีจำนวนช่องสีเท่าใด และโมเดลนี้มีคำตอบที่เป็นไปได้กี่แบบ

In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: ปรับโครงข่ายที่ฝึกมาแล้ว (ระบุตัวเลข 2 ค่า)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   นำโครงข่าย ResNet-18 ซึ่งฝึกมาแล้วกับภาพถ่ายทั่วไป (ImageNet) มาปรับใช้กับภาพถ่ายรังสีทรวงอก
#   ตามแนวคิด "การเรียนรู้แบบถ่ายโอน" (transfer learning) โดยปรับ 2 ส่วน
#     ก) ชั้นแรก (input layer)  : ให้รับภาพตามจำนวนช่องสี (channel) ของภาพถ่ายรังสี
#     ข) ชั้นสุดท้าย (output layer) : ให้ตอบได้ตามจำนวนคำตอบที่เป็นไปได้ของงานนี้
#   งานของผู้เรียน: แทนที่ ... (ช่องว่าง) ทั้ง 2 บรรทัดที่มี # TODO ด้วยตัวเลขที่ถูกต้อง
#   สิ่งที่ผู้เรียนจะเห็น: รายละเอียดของชั้นแรกและชั้นสุดท้ายหลังปรับแล้ว
#   (หากยังกรอกไม่ครบ เซลล์จะเกิดข้อผิดพลาด ซึ่งเป็นเรื่องปกติของแบบฝึกหัด)
# ============================================================================

# นำเข้าโมดูล models จากไลบรารี torchvision ซึ่งรวบรวมโครงข่ายสำเร็จรูปหลายแบบ
from torchvision import models
# สร้างโครงข่าย ResNet-18 (CNN ที่มี 18 ชั้น) แล้วเก็บไว้ในตัวแปร net
#   - models.resnet18(...)     : ฟังก์ชันสร้างโครงข่าย ResNet-18 จาก torchvision
#   - weights="IMAGENET1K_V1"  : โหลด "ค่าน้ำหนัก" (weights = ความรู้ที่โครงข่ายเรียนมา)
#                                ที่ฝึกกับ ImageNet (ภาพประมาณ 1 ล้านภาพ 1,000 คลาส) รุ่นที่ 1
net = models.resnet18(weights="IMAGENET1K_V1")   # ดาวน์โหลดโครงข่ายที่ฝึกมาแล้ว

# ----------------------------------------------------------------------------
# ส่วนที่ผู้เรียนต้องกรอก (2 ค่า)
# ----------------------------------------------------------------------------
# กำหนดจำนวนช่องสีของภาพที่จะป้อนเข้าโครงข่าย
#   - ... (Ellipsis) คือ "ช่องว่าง" สำหรับผู้เรียนเติมตัวเลข
#   - คำใบ้: ภาพสีมี 3 ช่อง (แดง เขียว น้ำเงิน) ให้พิจารณาว่าภาพถ่ายรังสีระดับเทามีกี่ช่อง
IN_CHANNELS = ...   # TODO: จำนวนช่องสีของภาพถ่ายรังสี (ภาพสี = 3, ภาพระดับเทา = ?)
# กำหนดจำนวนคลาส (class = กลุ่มคำตอบที่เป็นไปได้) ของงานนี้
#   - ... คือช่องว่างสำหรับผู้เรียนเติมตัวเลข
#   - คำใบ้: นับจำนวนคำตอบที่โมเดลต้องเลือก ตามที่ระบุในวงเล็บด้านขวา
N_CLASSES   = ...   # TODO: จำนวนคำตอบที่เป็นไปได้ (ปกติ / ปอดอักเสบ)

# ก) ชั้นแรกใหม่ที่รับภาพ IN_CHANNELS ช่อง โดยคงตัวกรองเดิมไว้ด้วยการรวมตัวกรอง RGB
# คัดลอกค่าน้ำหนักของ "ตัวกรอง" (filter) ในชั้นแรกเดิมเก็บไว้ใน old_w ก่อนจะแทนที่ชั้นนี้
#   - net.conv1        : ชั้นคอนโวลูชัน (convolution) ชั้นแรกของ ResNet-18
#   - .weight.data     : ค่าน้ำหนักของชั้นนั้น (รูปร่าง 64 ตัวกรอง × 3 ช่องสี × 7 × 7 พิกเซล)
#   - .clone()         : ทำสำเนาแยกออกมา เพื่อไม่ให้หายไปเมื่อแทนที่ชั้น
old_w = net.conv1.weight.data.clone()
# สร้างชั้นคอนโวลูชันใหม่ด้วย nn.Conv2d(...) จาก PyTorch แล้วนำไปแทนที่ชั้นแรกเดิม
#   - IN_CHANNELS   : จำนวนช่องสีของภาพที่รับเข้า (ค่าที่ผู้เรียนกรอก)
#   - 64            : จำนวนตัวกรองที่ส่งออก (เท่าเดิม เพื่อให้ต่อกับชั้นถัดไปได้)
#   - kernel_size=7 : ขนาดตัวกรอง 7×7 พิกเซล
#   - stride=2      : เลื่อนตัวกรองครั้งละ 2 พิกเซล (ภาพผลลัพธ์จึงเล็กลงครึ่งหนึ่ง)
#   - padding=3     : เติมขอบภาพรอบนอก 3 พิกเซล เพื่อให้ตัวกรองทำงานถึงขอบภาพได้
#   - bias=False    : ไม่ใช้ค่าคงที่บวกเพิ่ม (bias) เหมือนชั้นเดิมของ ResNet
#   (ทุกค่ายกเว้น IN_CHANNELS ตั้งให้ตรงกับชั้นเดิม จึงนำตัวกรองเดิมมาใช้ต่อได้)
net.conv1 = nn.Conv2d(IN_CHANNELS, 64, kernel_size=7, stride=2, padding=3, bias=False)
# if คือคำสั่งตรวจเงื่อนไข: ทำคำสั่งที่ย่อหน้าด้านล่างเฉพาะเมื่อเงื่อนไขเป็นจริง
#   - IN_CHANNELS == 1 : ตรวจว่าจำนวนช่องสีที่กำหนดเท่ากับ 1 หรือไม่ (== คือการเปรียบเทียบ "เท่ากับ")
if IN_CHANNELS == 1:
    # นำตัวกรองเดิมของ 3 ช่องสี (RGB) มารวมกันเป็นตัวกรองช่องเดียว แล้วใส่ให้ชั้นใหม่
    #   - old_w.sum(...)  : รวมค่าน้ำหนัก
    #   - dim=1           : รวมตามมิติที่ 1 คือมิติ "ช่องสี" (3 ช่อง -> 1 ช่อง)
    #   - keepdim=True    : คงมิตินั้นไว้ (ขนาดเป็น 1) ให้รูปร่างเป็น 64×1×7×7 ตรงกับชั้นใหม่
    #   วิธีนี้ทำให้ชั้นแรกยังคงความรู้เดิมจาก ImageNet แทนการเริ่มจากค่าสุ่ม
    net.conv1.weight.data = old_w.sum(dim=1, keepdim=True)
# ข) ชั้นสุดท้ายใหม่ที่มีเอาต์พุต N_CLASSES ค่า
# แทนที่ชั้นสุดท้าย (fc = fully connected layer ชั้นเชื่อมต่อเต็ม) ด้วยชั้นเชิงเส้นใหม่ nn.Linear(...)
#   - net.fc.in_features : จำนวนค่าที่เข้าชั้นนี้ (ของ ResNet-18 คือ 512) ใช้ค่าเดิมเพื่อให้ต่อกันได้
#   - N_CLASSES          : จำนวนค่าที่ส่งออก = จำนวนคลาส (ค่าที่ผู้เรียนกรอก) แทน 1,000 คลาสของ ImageNet
#   (ชั้นใหม่นี้เริ่มจากค่าสุ่ม และจะเรียนรู้ระหว่างการฝึก)
net.fc = nn.Linear(net.fc.in_features, N_CLASSES)

# ย้ายโครงข่ายไปไว้บนอุปกรณ์ที่เลือกไว้ (GPU หรือ CPU) ด้วย .to(device) แล้วเก็บกลับใน net
net = net.to(device)
# แสดงรายละเอียดของชั้นแรก (net.conv1) และชั้นสุดท้าย (net.fc) เพื่อตรวจสอบการปรับ
#   - sep="\n" : ใช้การขึ้นบรรทัดใหม่ (\n) คั่นระหว่างค่า แทนการเว้นวรรค จึงแสดงคนละบรรทัด
print(net.conv1, net.fc, sep="\n")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 1)
#   สร้าง ResNet-18 ที่ฝึกมาแล้วกับ ImageNet และปรับให้ใช้กับภาพถ่ายรังสีทรวงอก
#     - IN_CHANNELS = 1 : ภาพถ่ายรังสีเป็นภาพระดับเทา (grayscale) มีเพียง 1 ช่องสี
#     - N_CLASSES   = 2 : มีคำตอบที่เป็นไปได้ 2 แบบ คือ ปกติ (0) และ ปอดอักเสบ (1)
#   ผลลัพธ์: ตัวแปร net พร้อมสำหรับการฝึก และแสดงรายละเอียดของชั้นแรกและชั้นสุดท้าย
#   (รันเซลล์นี้ทับผลของแบบฝึกหัดได้ทันที)
# ============================================================================

# นำเข้าโมดูล models จากไลบรารี torchvision (รวมโครงข่ายสำเร็จรูป)
from torchvision import models
# สร้าง ResNet-18 พร้อมค่าน้ำหนักที่ฝึกกับ ImageNet แล้ว
#   - weights="IMAGENET1K_V1" : ใช้ค่าน้ำหนักชุดที่ฝึกกับ ImageNet-1K รุ่นที่ 1
net = models.resnet18(weights="IMAGENET1K_V1")

# คำตอบที่ 1: ภาพถ่ายรังสีระดับเทามี 1 ช่องสี (ภาพสีมี 3 ช่อง)
IN_CHANNELS = 1     # ภาพถ่ายรังสีระดับเทา = 1 ช่อง
# คำตอบที่ 2: มี 2 คลาส คือ ปกติ และ ปอดอักเสบ
N_CLASSES   = 2     # ปกติ / ปอดอักเสบ

# คัดลอกค่าน้ำหนักตัวกรองของชั้นแรกเดิม (64 × 3 × 7 × 7) เก็บไว้ใน old_w
#   - .weight.data : ค่าน้ำหนักของชั้น
#   - .clone()     : ทำสำเนาแยกไว้
old_w = net.conv1.weight.data.clone()
# สร้างชั้นคอนโวลูชันแรกใหม่ด้วย nn.Conv2d(...) ให้รับภาพ 1 ช่อง
#   - IN_CHANNELS (=1) : จำนวนช่องสีขาเข้า
#   - 64               : จำนวนตัวกรองขาออก (เท่าเดิม)
#   - kernel_size=7    : ตัวกรองขนาด 7×7
#   - stride=2         : เลื่อนครั้งละ 2 พิกเซล
#   - padding=3        : เติมขอบภาพ 3 พิกเซล
#   - bias=False       : ไม่ใช้ค่า bias (เหมือนชั้นเดิม)
net.conv1 = nn.Conv2d(IN_CHANNELS, 64, kernel_size=7, stride=2, padding=3, bias=False)
# ตรวจเงื่อนไข: หากภาพมี 1 ช่องสี (ซึ่งเป็นจริงในเฉลยนี้) ให้ทำบรรทัดถัดไป
if IN_CHANNELS == 1:
    # รวมตัวกรองของ 3 ช่องสีเดิมเป็นช่องเดียว แล้วใส่ให้ชั้นใหม่
    #   - dim=1        : รวมตามมิติช่องสี
    #   - keepdim=True : คงมิตินั้นไว้ ได้รูปร่าง 64×1×7×7
    net.conv1.weight.data = old_w.sum(dim=1, keepdim=True)
# แทนที่ชั้นสุดท้าย (fc) ด้วยชั้นเชิงเส้นใหม่ nn.Linear(...)
#   - net.fc.in_features : จำนวนค่าขาเข้าเดิม (512)
#   - N_CLASSES (=2)     : ส่งออก 2 ค่า คือคะแนนของ "ปกติ" และ "ปอดอักเสบ"
net.fc = nn.Linear(net.fc.in_features, N_CLASSES)

# ย้ายโครงข่ายไปไว้บนอุปกรณ์ที่เลือก (GPU หรือ CPU)
net = net.to(device)
# แสดงรายละเอียดชั้นแรกและชั้นสุดท้าย คนละบรรทัด (sep="\n" = คั่นด้วยการขึ้นบรรทัดใหม่)
print(net.conv1, net.fc, sep="\n")

💡 **คำอธิบายผลลัพธ์** โครงข่ายที่ฝึกมาแล้วได้รับการปรับ 2 ส่วน คือ ชั้นแรกรับภาพ 1 ช่อง (โดยเริ่มต้นจากตัวกรองภาพสีเดิม) และชั้นสุดท้ายมีเอาต์พุต 2 ค่า ส่วนชั้นกลางทั้งหมดยังคงความรู้ที่เรียนจาก ImageNet

## การฝึกโมเดล (โค้ดเตรียมไว้แล้ว ให้รันเซลล์)

หนึ่ง **รอบการฝึก (epoch)** หมายถึงการที่โมเดลประมวลผลภาพฝึกครบทุกภาพหนึ่งครั้ง และปรับพารามิเตอร์ภายในเพื่อลดความคลาดเคลื่อน ปฏิบัติการนี้ใช้เพียง **3 epoch** เพื่อจำกัดเวลา (ประมาณ 1 นาทีบน GPU หรือ 3–5 นาทีบน CPU)

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ฝึกโมเดล (training) ResNet-18 ที่ปรับแล้วให้แยกภาพถ่ายรังสี "ปกติ" กับ "ปอดอักเสบ"
#   โดยสร้างฟังก์ชัน 2 ตัว
#     - run_epoch(...)   : ประมวลผลข้อมูลครบ 1 รอบ (epoch) ทั้งแบบ "ฝึก" หรือ "ประเมินผลอย่างเดียว"
#     - train_model(...) : เรียก run_epoch ซ้ำหลายรอบ โดยฝึกด้วยชุดฝึก แล้ววัดผลด้วยชุดตรวจสอบ
#   ในแต่ละรอบ โมเดลจะทำนาย -> วัดความคลาดเคลื่อน (loss) -> ปรับค่าน้ำหนักให้ความคลาดเคลื่อนลดลง
#   สิ่งที่ผู้เรียนจะเห็น: ความแม่นยำ (accuracy) บนชุดฝึก (train_acc) และชุดตรวจสอบ (val_acc) ของแต่ละรอบ
#   (ใช้เวลาประมาณ 1 นาทีบน GPU หรือ 3–5 นาทีบน CPU)
# ============================================================================

# ลูปการฝึก (โค้ดเตรียมไว้แล้ว)
# นำเข้าโมดูล torch.optim ของ PyTorch (รวมวิธีปรับค่าน้ำหนัก หรือ optimizer) ตั้งชื่อย่อว่า optim
import torch.optim as optim
# กำหนดจำนวนรอบการฝึก (epoch) = 3 รอบ (1 รอบ = โมเดลเห็นภาพฝึกครบทุกภาพ 1 ครั้ง)
EPOCHS = 3

# ----------------------------------------------------------------------------
# ฟังก์ชัน run_epoch: ประมวลผลข้อมูล 1 รอบ แล้วคืนค่า (ความคลาดเคลื่อนเฉลี่ย, ความแม่นยำ)
#   - model    : โครงข่ายที่ใช้
#   - loader   : ตัวป้อนข้อมูล (DataLoader) ของชุดที่ต้องการ
#   - opt=None : optimizer สำหรับปรับค่าน้ำหนัก หากไม่ส่งมา (None = "ไม่มีค่า") จะประเมินผลอย่างเดียว
# ----------------------------------------------------------------------------
def run_epoch(model, loader, opt=None):
    """ประมวลผลข้อมูลหนึ่งรอบ: หากระบุ optimizer จะฝึกโมเดล หากไม่ระบุจะประเมินผลเท่านั้น"""
    # สร้างตัวแปร train เป็น True (ฝึก) หากมี optimizer ส่งมา หรือ False (ประเมินผล) หากไม่มี
    #   - is not None : ตรวจว่า "ไม่ใช่ค่าว่าง (None)"
    train = opt is not None
    # สร้างฟังก์ชันวัดความคลาดเคลื่อน (loss function) แบบ cross-entropy จาก PyTorch เก็บไว้ใน crit
    #   ใช้กับงานจำแนกกลุ่ม: ยิ่งโมเดลให้ความน่าจะเป็นกับคำตอบที่ถูกต่ำ ค่า loss ยิ่งสูง
    crit = nn.CrossEntropyLoss()
    # ตั้งโหมดของโมเดล: model.train(True) = โหมดฝึก, model.train(False) = โหมดประเมินผล
    #   (บางชั้น เช่น batch normalization ทำงานต่างกันในสองโหมดนี้)
    model.train(train)
    # ตั้งค่าเริ่มต้นของตัวนับ 3 ตัวพร้อมกัน (กำหนดหลายตัวแปรในบรรทัดเดียว)
    #   - total    = 0   : จำนวนภาพที่ประมวลผลแล้ว
    #   - correct  = 0   : จำนวนภาพที่ทำนายถูก
    #   - loss_sum = 0.0 : ผลรวมความคลาดเคลื่อน (เป็นทศนิยม)
    total, correct, loss_sum = 0, 0, 0.0
    # วนลูปดึงข้อมูลทีละชุดย่อย (batch) จากตัวป้อนข้อมูล
    #   - x : ภาพในชุดย่อยนั้น,  y : ป้ายกำกับที่ถูกต้องของภาพเหล่านั้น
    for x, y in loader:
        # เตรียมข้อมูลก่อนป้อนโมเดล
        #   - x.to(device)   : ย้ายภาพไปยัง GPU/CPU ที่เลือกไว้
        #   - y.view(-1)     : ปรับรูปร่างป้ายกำกับให้เป็นแถวเดียว (-1 = ให้คำนวณความยาวเอง)
        #   - .long()        : แปลงเป็นจำนวนเต็ม (รูปแบบที่ CrossEntropyLoss ต้องการ)
        #   - .to(device)    : ย้ายป้ายกำกับไปอุปกรณ์เดียวกับภาพ
        x, y = x.to(device), y.view(-1).long().to(device)
        # with คือการเปิด "บริบท" ชั่วคราวสำหรับคำสั่งที่ย่อหน้าอยู่ด้านใน
        #   - torch.set_grad_enabled(train) : เปิดการคำนวณ gradient (ทิศทางการปรับค่าน้ำหนัก) เฉพาะตอนฝึก
        #     ตอนประเมินผล (train=False) จะปิดไว้ เพื่อประหยัดหน่วยความจำและเวลา
        with torch.set_grad_enabled(train):
            # ป้อนภาพเข้าโมเดล ได้ผลลัพธ์ out เป็นคะแนน 2 ค่าต่อภาพ (ปกติ, ปอดอักเสบ)
            out = model(x)
            # คำนวณความคลาดเคลื่อน (loss) ระหว่างผลทำนาย out กับคำตอบที่ถูกต้อง y
            loss = crit(out, y)
            # หากอยู่ในโหมดฝึก ให้ปรับค่าน้ำหนักของโมเดล
            if train:
                # 3 คำสั่งในบรรทัดเดียว (คั่นด้วย ;) ทำตามลำดับ
                #   1) opt.zero_grad() : ล้างค่า gradient เดิมจากชุดย่อยก่อนหน้า
                #   2) loss.backward() : คำนวณ gradient ย้อนกลับ (backpropagation) ว่าควรปรับแต่ละค่าน้ำหนักอย่างไร
                #   3) opt.step()      : ปรับค่าน้ำหนักตาม gradient เพื่อให้ loss ลดลง
                opt.zero_grad(); loss.backward(); opt.step()
        # สะสมผลรวมความคลาดเคลื่อน (+= คือ "บวกเพิ่มเข้าไปในตัวแปรเดิม")
        #   - loss.item() : ดึงค่า loss ออกมาเป็นตัวเลขธรรมดา (ค่าเฉลี่ยของชุดย่อย)
        #   - * len(x)    : คูณด้วยจำนวนภาพในชุดย่อย เพื่อให้เป็นผลรวม
        loss_sum += loss.item() * len(x)
        # นับจำนวนที่ทายถูก และจำนวนภาพทั้งหมด (2 คำสั่งคั่นด้วย ;)
        #   - out.argmax(1)  : เลือกคลาสที่ได้คะแนนสูงสุดของแต่ละภาพ (ตามมิติที่ 1) = คำตอบของโมเดล
        #   - == y           : เทียบกับคำตอบจริง ได้ True (ถูก) / False (ผิด)
        #   - .sum().item()  : นับจำนวน True แล้วแปลงเป็นตัวเลขธรรมดา
        #   - total += len(x): เพิ่มจำนวนภาพที่ประมวลผลแล้ว
        correct += (out.argmax(1) == y).sum().item(); total += len(x)
    # ส่งผลลัพธ์กลับ (return) 2 ค่า: ความคลาดเคลื่อนเฉลี่ยต่อภาพ และความแม่นยำ (สัดส่วนที่ทายถูก)
    return loss_sum / total, correct / total

# ----------------------------------------------------------------------------
# ฟังก์ชัน train_model: ฝึกโมเดลหลายรอบ และแสดงความแม่นยำของแต่ละรอบ
#   - model        : โครงข่ายที่จะฝึก
#   - train_loader : ตัวป้อนข้อมูลชุดฝึก
#   - val_loader   : ตัวป้อนข้อมูลชุดตรวจสอบ
#   - epochs       : จำนวนรอบการฝึก
#   - lr=1e-3      : อัตราการเรียนรู้ (learning rate) = 0.001 (1e-3 คือ 1×10⁻³) ขนาดก้าวของการปรับค่าน้ำหนัก
# ----------------------------------------------------------------------------
def train_model(model, train_loader, val_loader, epochs, lr=1e-3):
    # สร้าง optimizer แบบ Adam (วิธีปรับค่าน้ำหนักที่นิยม ปรับขนาดก้าวให้อัตโนมัติ)
    #   - model.parameters() : ค่าน้ำหนักทั้งหมดของโมเดลที่จะให้ปรับ
    #   - lr=lr              : ใช้อัตราการเรียนรู้ตามที่ส่งเข้ามา
    opt = optim.Adam(model.parameters(), lr=lr)
    # วนลูปตามจำนวนรอบ: range(epochs) สร้างเลข 0, 1, ..., epochs-1 ให้ epoch ทีละค่า
    for epoch in range(epochs):
        # ฝึก 1 รอบด้วยชุดฝึก (ส่ง opt ไปด้วย จึงเป็นโหมดฝึก) ได้ tl = loss, ta = ความแม่นยำ ของชุดฝึก
        tl, ta = run_epoch(model, train_loader, opt)
        # ประเมินผลด้วยชุดตรวจสอบ (ไม่ส่ง opt จึงไม่ปรับโมเดล) ได้ vl = loss, va = ความแม่นยำ ของชุดตรวจสอบ
        vl, va = run_epoch(model, val_loader)
        # แสดงผลของรอบนี้ด้วย f-string
        #   - {epoch+1}/{epochs} : รอบที่เท่าไรจากทั้งหมด (+1 เพราะ Python นับเริ่มจาก 0)
        #   - {ta:.3f}, {va:.3f} : ความแม่นยำ แสดงเป็นทศนิยม (f) 3 ตำแหน่ง
        print(f"epoch {epoch+1}/{epochs}: train_acc={ta:.3f}  val_acc={va:.3f}")
    # ส่งโมเดลที่ฝึกแล้วกลับไป
    return model

# เริ่มฝึกจริง: เรียก train_model กับโมเดล net ชุดฝึก ชุดตรวจสอบ และจำนวนรอบ EPOCHS (=3)
#   แล้วเก็บโมเดลที่ฝึกแล้วกลับไว้ในตัวแปร net (lr ใช้ค่าเริ่มต้น 0.001)
net = train_model(net, train_loader, val_loader, EPOCHS)

💡 **คำอธิบายผลลัพธ์** *train_acc* คือความแม่นยำ (accuracy) บนภาพที่ใช้ฝึก ส่วน *val_acc* คือความแม่นยำบนภาพที่โมเดลไม่ได้ใช้ฝึก ซึ่งสะท้อนประสิทธิภาพจริงได้ดีกว่า หาก train_acc สูงกว่า val_acc อย่างมาก แสดงว่าโมเดลเกิด **การเรียนรู้เกิน (overfitting)** คือจดจำข้อมูลฝึกแทนการเรียนรู้รูปแบบทั่วไป ผลลัพธ์ของผู้เรียนแต่ละคนอาจแตกต่างกันเพียงเล็กน้อย เนื่องจากการฝึกมีองค์ประกอบเชิงสุ่ม

## การประเมินผลบนชุดทดสอบ (test set)

เซลล์ถัดไปให้โมเดลคำนวณ **ความน่าจะเป็นของภาวะปอดอักเสบ** สำหรับภาพทดสอบทั้ง 624 ภาพ โดยค่าเริ่มต้น ความน่าจะเป็นตั้งแต่ 0.5 ขึ้นไปหมายถึง "ทำนายว่าเป็นปอดอักเสบ"

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ให้โมเดลที่ฝึกแล้วทำนายภาพในชุดทดสอบ (test set) ทั้ง 624 ภาพ ซึ่งโมเดลไม่เคยเห็นระหว่างฝึก
#   ผลลัพธ์ที่ได้
#     - test_probs : ความน่าจะเป็นของภาวะปอดอักเสบ (0 ถึง 1) ของแต่ละภาพ
#     - y_true     : คำตอบที่ถูกต้อง (0 = ปกติ, 1 = ปอดอักเสบ)
#     - y_pred     : คำทำนายของโมเดล โดยใช้เกณฑ์ตัดสิน (threshold) 0.5
#   สิ่งที่ผู้เรียนจะเห็น: ความแม่นยำบนชุดทดสอบ เทียบกับความแม่นยำหากตอบ "ปอดอักเสบ" ทุกภาพ
# ============================================================================

# ให้โมเดลทำนายชุดทดสอบ (TEST) (โค้ดเตรียมไว้แล้ว)
# ตั้งโมเดลเป็นโหมดประเมินผล (evaluation) ด้วย net.eval() เพื่อไม่ให้บางชั้นทำงานแบบโหมดฝึก
net.eval()
# สร้าง list ว่าง [] 2 รายการ สำหรับเก็บความน่าจะเป็น (test_probs) และคำตอบจริง (y_true)
test_probs, y_true = [], []
# with torch.no_grad() : ปิดการคำนวณ gradient ชั่วคราว เพราะเป็นการทำนายอย่างเดียว ไม่ได้ฝึก (เร็วและประหยัดหน่วยความจำ)
with torch.no_grad():
    # วนลูปดึงภาพ (x) และป้ายกำกับ (y) ทีละชุดย่อยจากตัวป้อนข้อมูลชุดทดสอบ (เรียงตามลำดับเดิม)
    for x, y in test_loader:
        # คำนวณความน่าจะเป็นของปอดอักเสบ แล้วต่อท้ายเข้าไปใน test_probs (+= กับ list = ต่อท้ายรายการ)
        #   ทำตามลำดับจากในออกนอก:
        #   1) x.to(device)              : ย้ายภาพไปยัง GPU/CPU
        #   2) net(...)                  : ให้โมเดลทำนาย ได้คะแนน 2 ค่าต่อภาพ
        #   3) torch.softmax(..., dim=1) : แปลงคะแนนเป็นความน่าจะเป็นที่รวมกันได้ 1 (คำนวณตามมิติที่ 1 = มิติคลาส)
        #   4) [:, 1]                    : เลือกทุกภาพ (:) เฉพาะคอลัมน์ที่ 1 = ความน่าจะเป็นของ "ปอดอักเสบ"
        #   5) .cpu()                    : ย้ายผลกลับมาที่ CPU
        #   6) .tolist()                 : แปลงเป็น list ของ Python
        test_probs += torch.softmax(net(x.to(device)), dim=1)[:, 1].cpu().tolist()
        # เก็บคำตอบจริงต่อท้าย y_true
        #   - y.view(-1) : ปรับให้เป็นแถวเดียว,  .long() : จำนวนเต็ม,  .tolist() : แปลงเป็น list
        y_true += y.view(-1).long().tolist()
# แปลง list ทั้งสองให้เป็น NumPy array ด้วย np.array(...) เพื่อคำนวณเปรียบเทียบได้ทั้งชุดในคำสั่งเดียว
test_probs, y_true = np.array(test_probs), np.array(y_true)
# ตัดสินคำทำนายด้วยเกณฑ์ 0.5
#   - test_probs >= 0.5 : ได้ True หากความน่าจะเป็นตั้งแต่ 0.5 ขึ้นไป มิฉะนั้น False
#   - .astype(int)      : แปลง True/False เป็น 1/0 (1 = ทำนายว่าปอดอักเสบ, 0 = ทำนายว่าปกติ)
y_pred = (test_probs >= 0.5).astype(int)
# แสดงความแม่นยำบนชุดทดสอบ
#   - (y_pred == y_true).mean() : เทียบคำทำนายกับคำตอบจริงทีละภาพ แล้วหาค่าเฉลี่ยของ True (=1)
#                                  ได้สัดส่วนภาพที่ทายถูก (ความแม่นยำ, accuracy)
#   - y_true.mean()             : สัดส่วนภาพปอดอักเสบจริง = ความแม่นยำที่จะได้หากตอบ "ปอดอักเสบ" ทุกภาพ
#   - :.3f                      : แสดงทศนิยม 3 ตำแหน่ง
print(f"ความแม่นยำบนชุดทดสอบ: {(y_pred == y_true).mean():.3f}   (หากตอบ 'ปอดอักเสบ' ทุกภาพ จะได้ {y_true.mean():.3f})")

💡 **ข้อสังเกต** ความแม่นยำบนชุดทดสอบมักต่ำกว่า val_acc อย่างชัดเจน เนื่องจากชุดทดสอบของ PneumoniaMNIST มาจากกลุ่มผู้ป่วยที่แตกต่างจากชุดฝึกและมีสัดส่วนภาพปกติมากกว่า สถานการณ์นี้เทียบได้กับการนำโมเดลไปใช้ในโรงพยาบาลอื่น จึงจำเป็นต้องประเมินโมเดลกับข้อมูลใหม่ที่แยกออกมาอย่างแท้จริงเสมอ

### การอ่านเมทริกซ์ความสับสน (confusion matrix)

**Confusion matrix** แสดงจำนวนผลลัพธ์ 4 ประเภท (ผลบวก (positive) = ปอดอักเสบ)

| | โมเดลทำนายว่า **ปกติ** | โมเดลทำนายว่า **ปอดอักเสบ** |
|---|---|---|
| **ปกติจริง** | TN — ผลลบจริง (true negative) ✔ | FP — ผลบวกลวง (false positive) |
| **ปอดอักเสบจริง** | FN — ผลลบลวง (false negative) คือ **ผู้ป่วยที่ตรวจไม่พบ** | TP — ผลบวกจริง (true positive) ✔ |

- **ความไว (sensitivity, recall)** = สัดส่วนของผู้ป่วยปอดอักเสบจริงที่โมเดลตรวจพบ = TP / (TP + FN)
- **ความจำเพาะ (specificity)** = สัดส่วนของผู้ที่ปกติจริงที่โมเดลทำนายว่าปกติได้ถูกต้อง = TN / (TN + FP)

**⭐ แบบฝึกหัดที่ 2 (Exercise 2):** ให้เขียนสูตร 2 บรรทัดท้ายเซลล์ โดยใช้ตัวแปร `tp`, `tn`, `fp`, `fn`
*คำใบ้:* สามารถคัดลอกสูตรจากรายการข้างต้น เช่น `sensitivity = tp / (tp + fn)`

In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: ความไวและความจำเพาะจาก confusion matrix
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   สร้างเมทริกซ์ความสับสน (confusion matrix) จากผลการทำนายชุดทดสอบ เพื่อดูว่าโมเดลผิดพลาด "แบบใด"
#     - TN ผลลบจริง (true negative)   : ปกติจริง และทำนายว่าปกติ
#     - FP ผลบวกลวง (false positive)  : ปกติจริง แต่ทำนายว่าปอดอักเสบ
#     - FN ผลลบลวง (false negative)   : ปอดอักเสบจริง แต่ทำนายว่าปกติ (ผู้ป่วยที่ตรวจไม่พบ)
#     - TP ผลบวกจริง (true positive)  : ปอดอักเสบจริง และทำนายว่าปอดอักเสบ
#   แล้วคำนวณ ความไว (sensitivity) และ ความจำเพาะ (specificity)
#   งานของผู้เรียน: แทนที่ ... ใน 2 บรรทัดที่มี # TODO ด้วยสูตรตามคำใบ้ โดยใช้ตัวแปร tn, fp, fn, tp
#   สิ่งที่ผู้เรียนจะเห็น: ภาพ confusion matrix, ค่า TN/FP/FN/TP และค่าความไว/ความจำเพาะ
# ============================================================================
# นำเข้าเครื่องมือ 2 ตัวจากโมดูล sklearn.metrics ของไลบรารี scikit-learn (ไลบรารีการเรียนรู้ของเครื่อง)
#   - confusion_matrix       : ฟังก์ชันนับจำนวน TN, FP, FN, TP เป็นตาราง
#   - ConfusionMatrixDisplay : คลาสสำหรับวาด confusion matrix เป็นภาพ
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# สร้าง confusion matrix เก็บไว้ใน cm (ตาราง 2×2: แถว = คำตอบจริง, คอลัมน์ = คำทำนาย)
#   - y_true        : คำตอบจริงของภาพทดสอบ
#   - y_pred        : คำทำนายของโมเดล (เกณฑ์ 0.5)
#   - labels=[0, 1] : กำหนดลำดับคลาส 0 (ปกติ) ก่อน 1 (ปอดอักเสบ) เสมอ
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
# วาด confusion matrix เป็นภาพ (2 ขั้นต่อกัน)
#   1) ConfusionMatrixDisplay(cm, display_labels=[...]) : เตรียมภาพจากตาราง cm
#        - display_labels=["normal", "pneumonia"] : ชื่อคลาสที่แสดงบนแกน (ภาษาอังกฤษ เพราะฟอนต์กราฟไม่รองรับภาษาไทย)
#   2) .plot(...) : วาดภาพ
#        - cmap="Blues"     : ใช้โทนสีน้ำเงิน ช่องที่มีจำนวนมากจะสีเข้ม
#        - values_format="d": แสดงตัวเลขในช่องเป็นจำนวนเต็ม (d = decimal integer)
ConfusionMatrixDisplay(cm, display_labels=["normal", "pneumonia"]).plot(cmap="Blues", values_format="d")
# 3 คำสั่งในบรรทัดเดียว (คั่นด้วย ;)
#   - plt.xlabel(...) : ตั้งชื่อแกนนอน = คำทำนายของโมเดล
#   - plt.ylabel(...) : ตั้งชื่อแกนตั้ง = การวินิจฉัยจริง
#   - plt.show()      : แสดงภาพ
plt.xlabel("Model's prediction"); plt.ylabel("True diagnosis"); plt.show()

# แตกค่าในตาราง 2×2 ออกเป็น 4 ตัวแปร
#   - cm.ravel() : เรียงตารางให้เป็นแถวเดียว ตามลำดับ [แถวบนซ้าย, บนขวา, ล่างซ้าย, ล่างขวา]
#                  = [TN, FP, FN, TP] แล้วกระจายใส่ tn, fp, fn, tp ตามลำดับ
tn, fp, fn, tp = cm.ravel()
# แสดงจำนวนของแต่ละประเภทด้วย f-string
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

# ----------------------------------------------------------------------------
# ส่วนที่ผู้เรียนต้องกรอก
# ----------------------------------------------------------------------------
# คำนวณความไว (sensitivity) = สัดส่วนผู้ป่วยปอดอักเสบจริงที่โมเดลตรวจพบ
#   - ... คือช่องว่างให้ผู้เรียนเขียนสูตร โดยแปลงสูตรในคำใบ้ด้านขวาให้ใช้ชื่อตัวแปรตัวพิมพ์เล็กที่สร้างไว้ด้านบน
sensitivity = ...   # TODO: TP / (TP + FN)
# คำนวณความจำเพาะ (specificity) = สัดส่วนคนปกติจริงที่โมเดลระบุว่าปกติได้ถูกต้อง
#   - ... คือช่องว่างให้ผู้เรียนเขียนสูตรตามคำใบ้ด้านขวาในลักษณะเดียวกัน
specificity = ...   # TODO: TN / (TN + FP)
# แสดงค่าความไวและความจำเพาะ เป็นทศนิยม 3 ตำแหน่ง (:.3f)
#   (หากยังไม่ได้กรอก ... บรรทัดนี้จะเกิดข้อผิดพลาด เพราะไม่สามารถแสดง ... เป็นตัวเลขได้)
print(f"ความไว (sensitivity)={sensitivity:.3f}  ความจำเพาะ (specificity)={specificity:.3f}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 2)
#   สร้าง confusion matrix ของชุดทดสอบ แล้วคำนวณ
#     - ความไว (sensitivity)     = TP / (TP + FN) : ในผู้ป่วยปอดอักเสบจริงทั้งหมด โมเดลตรวจพบกี่ส่วน
#     - ความจำเพาะ (specificity) = TN / (TN + FP) : ในคนปกติจริงทั้งหมด โมเดลระบุว่าปกติได้กี่ส่วน
#   สิ่งที่ผู้เรียนจะเห็น: ภาพ confusion matrix, ค่า TN/FP/FN/TP และค่าความไว/ความจำเพาะ
# ============================================================================
# นำเข้า confusion_matrix (นับ TN/FP/FN/TP) และ ConfusionMatrixDisplay (วาดภาพ) จาก scikit-learn
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# สร้าง confusion matrix (แถว = คำตอบจริง, คอลัมน์ = คำทำนาย)
#   - y_true, y_pred : คำตอบจริง และคำทำนายของโมเดล
#   - labels=[0, 1]  : เรียงคลาส ปกติ (0) ก่อน ปอดอักเสบ (1)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
# วาด confusion matrix
#   - display_labels=["normal", "pneumonia"] : ชื่อคลาสบนแกน
#   - cmap="Blues"                           : โทนสีน้ำเงิน
#   - values_format="d"                      : แสดงตัวเลขเป็นจำนวนเต็ม
ConfusionMatrixDisplay(cm, display_labels=["normal", "pneumonia"]).plot(cmap="Blues", values_format="d")
# ตั้งชื่อแกนนอน (คำทำนาย) แกนตั้ง (การวินิจฉัยจริง) แล้วแสดงภาพ (3 คำสั่งคั่นด้วย ;)
plt.xlabel("Model's prediction"); plt.ylabel("True diagnosis"); plt.show()

# แตกตาราง 2×2 เป็น 4 ตัวแปรตามลำดับ TN, FP, FN, TP ด้วย cm.ravel()
tn, fp, fn, tp = cm.ravel()
# แสดงจำนวนของแต่ละประเภท
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

# เฉลย: ความไว = ผลบวกจริง / (ผลบวกจริง + ผลลบลวง) = จำนวนที่ตรวจพบ / จำนวนผู้ป่วยปอดอักเสบจริงทั้งหมด
sensitivity = tp / (tp + fn)
# เฉลย: ความจำเพาะ = ผลลบจริง / (ผลลบจริง + ผลบวกลวง) = จำนวนที่ระบุว่าปกติถูก / จำนวนคนปกติจริงทั้งหมด
specificity = tn / (tn + fp)
# แสดงค่าความไวและความจำเพาะ เป็นทศนิยม 3 ตำแหน่ง (:.3f)
print(f"ความไว (sensitivity)={sensitivity:.3f}  ความจำเพาะ (specificity)={specificity:.3f}")

💡 **คำอธิบายผลลัพธ์** ความแม่นยำเพียงค่าเดียวไม่ได้แสดงว่าโมเดลผิดพลาด *ในลักษณะใด* ขณะที่ confusion matrix แสดงได้ชัดเจน ผลที่พบบ่อยกับชุดข้อมูลนี้คือความไวสูงมากแต่ความจำเพาะต่ำกว่า กล่าวคือ โมเดลมีแนวโน้มทำนายว่าเป็นปอดอักเสบมากเกินจริง ส่วนหนึ่งเนื่องจากภาพฝึกส่วนใหญ่เป็นภาพปอดอักเสบ

✍️ **คำถามที่ 1** จาก confusion matrix ข้างต้น (ก) โปรดระบุจำนวนผู้ป่วยเด็กปอดอักเสบที่โมเดล **ตรวจไม่พบ** (ข) โปรดระบุจำนวนเด็กปกติที่ได้รับ **ผลบวกลวง** (ค) จงอธิบายผลที่จะเกิดขึ้นกับผู้ป่วยในแต่ละกรณีในเวชปฏิบัติจริง และระบุว่าความผิดพลาดประเภทใดมีความรุนแรงมากกว่า เพราะเหตุใด

✍️ **คำตอบ:**



## การพิจารณาภาพที่โมเดลจำแนกผิด (โค้ดเตรียมไว้แล้ว ให้รันเซลล์)

ตัวเลขเชิงสถิติเพียงอย่างเดียวไม่เพียงพอ บุคลากรทางการแพทย์ควรพิจารณาด้วยว่าโมเดลผิดพลาดกับภาพ *ใด* เซลล์ด้านล่างแสดง **ผู้ป่วยปอดอักเสบที่โมเดลตรวจไม่พบ** (ผลลบลวง) และ **ผลบวกลวง** ที่โมเดลมีความมั่นใจสูงที่สุด พร้อมความน่าจะเป็นของภาวะปอดอักเสบที่โมเดลคำนวณ (`P(pneu)`) และแสดงภาพที่จำแนกถูกต้องเพื่อเปรียบเทียบ หากต้องการแสดงภาพเพิ่ม ให้เปลี่ยนค่า `N_SHOW`

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   แสดง "ภาพจริง" ที่โมเดลจำแนกผิด เพื่อให้บุคลากรทางการแพทย์พิจารณาว่าโมเดลพลาดกับภาพลักษณะใด
#     1) ผู้ป่วยปอดอักเสบที่โมเดลตรวจไม่พบ (ผลลบลวง, false negative) — เรียงจากที่โมเดลมั่นใจผิดมากที่สุด
#     2) ผลบวกลวง (false positive) — เด็กปกติที่โมเดลบอกว่าปอดอักเสบ เรียงจากที่มั่นใจผิดมากที่สุด
#     3) และ 4) ภาพที่โมเดลจำแนกถูก (ทั้งปอดอักเสบและปกติ) เพื่อใช้เปรียบเทียบ
#   แต่ละภาพจะแสดงลำดับภาพ (#) และความน่าจะเป็นของปอดอักเสบที่โมเดลคำนวณ P(pneu)
#   ผู้เรียนเปลี่ยนจำนวนภาพที่แสดงได้ที่ตัวแปร N_SHOW
# ============================================================================

# แสดงภาพที่โมเดลจำแนกผิดโดยมีความมั่นใจสูงที่สุด (โค้ดเตรียมไว้แล้ว)
# กำหนดจำนวนภาพสูงสุดที่จะแสดงในแต่ละแถว = 8 ภาพ
N_SHOW = 8

# ----------------------------------------------------------------------------
# ฟังก์ชัน show_cases: แสดงภาพจากชุดทดสอบตามลำดับภาพที่กำหนด เรียงเป็น 1 แถว
#   - idx   : ลำดับ (ดัชนี, index) ของภาพที่ต้องการแสดง
#   - title : หัวเรื่องของแถวภาพ
# ----------------------------------------------------------------------------
def show_cases(idx, title):
    # แปลง idx เป็น list แล้วตัดเอาเฉพาะ N_SHOW ตัวแรก
    #   - list(...)  : แปลงเป็น list ของ Python
    #   - [:N_SHOW]  : การตัดช่วง (slicing) เอาสมาชิกตั้งแต่ตัวแรกถึงก่อนตำแหน่ง N_SHOW
    idx = list(idx)[:N_SHOW]
    # if not idx : หาก list ว่าง (ไม่มีภาพในกลุ่มนี้เลย) ให้ทำคำสั่งที่ย่อหน้าด้านล่าง
    if not idx:
        # แสดงข้อความว่าไม่พบกรณีดังกล่าว แล้ว return (ออกจากฟังก์ชันทันที ไม่วาดภาพ) — 2 คำสั่งคั่นด้วย ;
        print(f"{title}: ไม่พบกรณีดังกล่าว"); return
    # สร้างพื้นที่วาดภาพ 1 แถว จำนวนคอลัมน์เท่ากับจำนวนภาพ
    #   - 1, len(idx)                 : 1 แถว × len(idx) คอลัมน์
    #   - figsize=(2 * len(idx), 2.6) : กว้างภาพละ 2 นิ้ว สูง 2.6 นิ้ว
    #   - squeeze=False               : ให้ axes เป็นตาราง 2 มิติเสมอ (แม้มีภาพเดียว) จึงเรียก axes[0] ได้ทุกกรณี
    fig, axes = plt.subplots(1, len(idx), figsize=(2 * len(idx), 2.6), squeeze=False)
    # วนลูปจับคู่ช่องภาพในแถวแรก (axes[0]) กับลำดับภาพ i ทีละคู่
    for ax, i in zip(axes[0], idx):
        # แสดงภาพที่ i ของชุดทดสอบ
        #   - test_ds[i]   : ข้อมูลรายการที่ i ได้ (ภาพ, ป้ายกำกับ)
        #   - [0]          : เลือกเฉพาะภาพ
        #   - .squeeze()   : ตัดมิติขนาด 1 ออก ให้เป็นภาพ 2 มิติ
        #   - cmap="gray"  : แสดงเป็นโทนสีเทา
        ax.imshow(test_ds[i][0].squeeze(), cmap="gray")
        # ตั้งหัวภาพเป็น ลำดับภาพ และความน่าจะเป็นของปอดอักเสบ (ทศนิยม 2 ตำแหน่ง, :.2f)
        #   - fontsize=8 : ขนาดตัวอักษร 8
        ax.set_title(f"#{i}  P(pneu)={test_probs[i]:.2f}", fontsize=8)
        # ซ่อนแกนรอบภาพ
        ax.axis("off")
    # ตั้งหัวเรื่องรวมของทั้งแถวด้วย fig.suptitle(title) แล้วแสดงภาพด้วย plt.show() (2 คำสั่งคั่นด้วย ;)
    fig.suptitle(title); plt.show()

# หาลำดับภาพที่เป็นผลลบลวง (FN): ปอดอักเสบจริง (y_true == 1) และ (&) ทำนายว่าปกติ (y_pred == 0)
#   - np.where(เงื่อนไข) : ฟังก์ชันของ NumPy คืนตำแหน่งที่เงื่อนไขเป็นจริง
#   - [0]                : np.where คืนผลเป็นชุด (tuple) จึงเลือกชุดแรกซึ่งเป็นลำดับภาพ
#   - ต้องใส่วงเล็บรอบแต่ละเงื่อนไขเมื่อใช้ & ร่วมกับ ==
fn_idx = np.where((y_true == 1) & (y_pred == 0))[0]
# หาลำดับภาพที่เป็นผลบวกลวง (FP): ปกติจริง (y_true == 0) แต่ทำนายว่าปอดอักเสบ (y_pred == 1)
fp_idx = np.where((y_true == 0) & (y_pred == 1))[0]
# แสดงผู้ป่วยที่ตรวจไม่พบ เรียงจากความน่าจะเป็นต่ำสุด (โมเดลมั่นใจว่า "ปกติ" มากที่สุด = ผิดแบบมั่นใจที่สุด)
#   - test_probs[fn_idx]       : ความน่าจะเป็นของภาพกลุ่ม FN
#   - np.argsort(...)          : คืนลำดับที่ทำให้ค่าเรียงจากน้อยไปมาก
#   - fn_idx[...]              : จัดเรียงลำดับภาพ FN ตามนั้น
#   - f"... {len(fn_idx)} ..." : หัวเรื่อง พร้อมจำนวนภาพ FN ทั้งหมด
show_cases(fn_idx[np.argsort(test_probs[fn_idx])],  f"Missed pneumonia (false negatives): {len(fn_idx)} in total")
# แสดงผลบวกลวง เรียงจากความน่าจะเป็นสูงสุด
#   - -test_probs[fp_idx] : ใส่เครื่องหมายลบ เพื่อให้ argsort เรียงจากมากไปน้อย
show_cases(fp_idx[np.argsort(-test_probs[fp_idx])], f"False alarms (false positives): {len(fp_idx)} in total")
# หาลำดับภาพผลบวกจริง (TP): ปอดอักเสบจริง และทำนายว่าปอดอักเสบ
tp_idx = np.where((y_true == 1) & (y_pred == 1))[0]
# หาลำดับภาพผลลบจริง (TN): ปกติจริง และทำนายว่าปกติ
tn_idx = np.where((y_true == 0) & (y_pred == 0))[0]
# แสดงภาพปอดอักเสบที่ตรวจพบถูกต้อง เรียงจากที่มั่นใจมากที่สุด (ความน่าจะเป็นสูงสุดก่อน) เพื่อเปรียบเทียบ
show_cases(tp_idx[np.argsort(-test_probs[tp_idx])], "For comparison - correctly detected pneumonia")
# แสดงภาพปกติที่ระบุถูกต้อง เรียงจากความน่าจะเป็นปอดอักเสบต่ำสุด (มั่นใจว่าปกติมากที่สุด) เพื่อเปรียบเทียบ
show_cases(tn_idx[np.argsort(test_probs[tn_idx])],  "For comparison - correctly cleared normals")

✍️ **คำถามที่ 2** จากการเปรียบเทียบภาพที่จำแนกผิดกับภาพที่จำแนกถูกต้อง ผู้เรียนพิจารณาว่าปัจจัยใดอาจทำให้โมเดลผิดพลาด (เช่น คุณภาพภาพ การจัดท่าผู้ป่วย ความผิดปกติที่ไม่ชัดเจน หรือความละเอียดที่ต่ำมากเพียง 28×28 พิกเซล) และผู้เรียนจะเชื่อถือความน่าจะเป็น 0.95 มากกว่า 0.55 หรือไม่ เพราะเหตุใด

✍️ **คำตอบ:**



## การเลือกเกณฑ์ตัดสิน (threshold)

โมเดลให้ผลลัพธ์เป็นความน่าจะเป็น ส่วนจุดตัดเป็นสิ่งที่ผู้ใช้งานกำหนด การลดเกณฑ์ตัดสิน (เช่น 0.3) ทำให้ภาพจำนวนมากขึ้นถูกจำแนกว่าเป็นปอดอักเสบ ส่งผลให้ **ความไวเพิ่มขึ้นและความจำเพาะลดลง** ส่วนการเพิ่มเกณฑ์ (เช่น 0.8) ให้ผลในทางตรงข้าม ค่าที่เหมาะสมขึ้นกับวัตถุประสงค์ทางคลินิก เช่น การคัดกรองหรือการคัดแยกผู้ป่วย (screening/triage) เทียบกับการยืนยันการวินิจฉัย

**⭐ แบบฝึกหัดที่ 3 (Exercise 3):** ให้กำหนด `THRESHOLD` เป็นค่าระหว่าง 0 ถึง 1 (ทดลองค่า 0.3 จากนั้นทดลองค่า 0.8 หรือ 0.9) แล้วรันเซลล์ใหม่ และสังเกตการเปลี่ยนแปลงของ confusion matrix
*คำใบ้:* ให้ระบุตัวเลข เช่น `THRESHOLD = 0.3`

In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: เปลี่ยนเกณฑ์ตัดสิน (threshold)
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   ทดลองเปลี่ยน "เกณฑ์ตัดสิน" (threshold) ซึ่งเป็นจุดตัดของความน่าจะเป็นที่ถือว่า "ปอดอักเสบ"
#   แล้วดูผลต่อ confusion matrix, จำนวนผู้ป่วยที่ตรวจไม่พบ (FN), ผลบวกลวง (FP),
#   ความไว (sensitivity) และความจำเพาะ (specificity)
#   ในทางคลินิก การเลือกเกณฑ์ขึ้นกับวัตถุประสงค์ เช่น การคัดกรอง (screening) เทียบกับการยืนยันการวินิจฉัย
#   งานของผู้เรียน: แทนที่ ... ด้วยตัวเลขระหว่าง 0 ถึง 1 แล้วรันเซลล์ซ้ำหลายค่าเพื่อเปรียบเทียบ
#   สิ่งที่ผู้เรียนจะเห็น: ภาพ confusion matrix ของเกณฑ์ที่เลือก และสรุป FN, FP, ความไว, ความจำเพาะ
# ============================================================================
# กำหนดเกณฑ์ตัดสิน
#   - ... คือช่องว่างให้ผู้เรียนเติมตัวเลขระหว่าง 0 ถึง 1 (ค่าเริ่มต้นที่ใช้ก่อนหน้านี้คือ 0.5)
THRESHOLD = ...   # TODO: ตัวเลขระหว่าง 0 ถึง 1 (ค่าเริ่มต้นคือ 0.5)

# ตัดสินคำทำนายใหม่ตามเกณฑ์ที่เลือก
#   - test_probs >= THRESHOLD : True หากความน่าจะเป็นถึงเกณฑ์ (ทำนายว่าปอดอักเสบ)
#   - .astype(int)            : แปลง True/False เป็น 1/0
y_pred_t = (test_probs >= THRESHOLD).astype(int)
# สร้าง confusion matrix ของคำทำนายชุดใหม่ (labels=[0, 1] เรียงคลาส ปกติ ก่อน ปอดอักเสบ)
cm_t = confusion_matrix(y_true, y_pred_t, labels=[0, 1])
# วาด confusion matrix
#   - display_labels : ชื่อคลาสบนแกน,  cmap="Blues" : โทนสีน้ำเงิน,  values_format="d" : แสดงจำนวนเต็ม
ConfusionMatrixDisplay(cm_t, display_labels=["normal", "pneumonia"]).plot(cmap="Blues", values_format="d")
# 4 คำสั่งคั่นด้วย ;
#   - plt.title(f"Threshold = {THRESHOLD}") : ตั้งหัวภาพแสดงเกณฑ์ที่ใช้
#   - plt.xlabel(...) / plt.ylabel(...)     : ชื่อแกนนอน (คำทำนาย) และแกนตั้ง (การวินิจฉัยจริง)
#   - plt.show()                            : แสดงภาพ
plt.title(f"Threshold = {THRESHOLD}"); plt.xlabel("Model's prediction"); plt.ylabel("True diagnosis"); plt.show()
# แตกตารางเป็น 4 ตัวแปร ตามลำดับ TN, FP, FN, TP (ต่อท้ายชื่อด้วย _t เพื่อแยกจากชุดเกณฑ์ 0.5)
tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
# แสดงสรุปผล (คำสั่งเดียวที่เขียนต่อ 2 บรรทัด: ข้อความ f-string 2 ส่วนที่อยู่ติดกันจะถูกต่อเป็นข้อความเดียว)
#   - {fn_t}, {fp_t}                 : จำนวนผู้ป่วยที่ตรวจไม่พบ และจำนวนผลบวกลวง
#   - {tp_t/(tp_t+fn_t):.3f}         : ความไว = TP / (TP + FN) ทศนิยม 3 ตำแหน่ง
#   - {tn_t/(tn_t+fp_t):.3f}         : ความจำเพาะ = TN / (TN + FP) ทศนิยม 3 ตำแหน่ง
print(f"เกณฑ์ {THRESHOLD}: ตรวจไม่พบ (FN)={fn_t}, ผลบวกลวง (FP)={fp_t}, "
      f"ความไว={tp_t/(tp_t+fn_t):.3f}, ความจำเพาะ={tn_t/(tn_t+fp_t):.3f}")

In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (เฉลยแบบฝึกหัดที่ 3)
#   ตัวอย่างการใช้เกณฑ์ตัดสิน (threshold) = 0.3 ซึ่งต่ำกว่าค่าเริ่มต้น 0.5
#   เกณฑ์ที่ต่ำลงทำให้ภาพจำนวนมากขึ้นถูกจัดเป็น "ปอดอักเสบ" ผลคือ ความไวเพิ่มขึ้น (ตรวจไม่พบน้อยลง)
#   แต่ความจำเพาะลดลง (ผลบวกลวงมากขึ้น) เหมาะกับงานคัดกรองที่ไม่อยากพลาดผู้ป่วย
#   ผู้เรียนลองเปลี่ยนเป็น 0.8 หรือ 0.9 เพื่อดูผลในทางตรงข้ามได้
#   สิ่งที่ผู้เรียนจะเห็น: ภาพ confusion matrix และสรุป FN, FP, ความไว, ความจำเพาะ
# ============================================================================
# เฉลย: กำหนดเกณฑ์ตัดสิน = 0.3
THRESHOLD = 0.3   # เกณฑ์ที่ต่ำลง ซึ่งอาจใช้ในการคัดกรอง (เพื่อตรวจพบผู้ป่วยได้มากขึ้น)

# ตัดสินคำทำนายใหม่: ความน่าจะเป็นตั้งแต่ 0.3 ขึ้นไป = 1 (ปอดอักเสบ) มิฉะนั้น = 0 (ปกติ)
#   - .astype(int) : แปลง True/False เป็น 1/0
y_pred_t = (test_probs >= THRESHOLD).astype(int)
# สร้าง confusion matrix ของคำทำนายชุดใหม่ (labels=[0, 1] เรียงคลาส ปกติ ก่อน ปอดอักเสบ)
cm_t = confusion_matrix(y_true, y_pred_t, labels=[0, 1])
# วาด confusion matrix (ชื่อคลาสบนแกน, โทนสีน้ำเงิน, ตัวเลขจำนวนเต็ม)
ConfusionMatrixDisplay(cm_t, display_labels=["normal", "pneumonia"]).plot(cmap="Blues", values_format="d")
# ตั้งหัวภาพ (แสดงเกณฑ์) ชื่อแกนนอน ชื่อแกนตั้ง แล้วแสดงภาพ (4 คำสั่งคั่นด้วย ;)
plt.title(f"Threshold = {THRESHOLD}"); plt.xlabel("Model's prediction"); plt.ylabel("True diagnosis"); plt.show()
# แตกตารางเป็น 4 ตัวแปร ตามลำดับ TN, FP, FN, TP
tn_t, fp_t, fn_t, tp_t = cm_t.ravel()
# แสดงสรุป: จำนวน FN, FP, ความไว = TP/(TP+FN) และความจำเพาะ = TN/(TN+FP) ทศนิยม 3 ตำแหน่ง
#   (คำสั่งเดียวเขียนต่อ 2 บรรทัด ข้อความ 2 ส่วนจะถูกต่อกันเป็นข้อความเดียว)
print(f"เกณฑ์ {THRESHOLD}: ตรวจไม่พบ (FN)={fn_t}, ผลบวกลวง (FP)={fp_t}, "
      f"ความไว={tp_t/(tp_t+fn_t):.3f}, ความจำเพาะ={tn_t/(tn_t+fp_t):.3f}")

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   แสดงความสัมพันธ์เชิงแลกเปลี่ยน (trade-off) ระหว่างความไวและความจำเพาะเมื่อเปลี่ยนเกณฑ์ตัดสิน
#   โดยแบ่งเป็น 2 ส่วน
#     1) ตาราง: คำนวณ FN, FP, ความไว และความจำเพาะ ของเกณฑ์ตัดสินหลายค่า (0.1 ถึง 0.95)
#     2) เส้นโค้ง ROC (ROC curve) ซึ่งแสดงความไว เทียบกับ (1 - ความจำเพาะ) ของ "ทุก" เกณฑ์
#        พร้อมพื้นที่ใต้เส้นโค้ง (AUC) ที่สรุปความสามารถในการแยกสองกลุ่ม (1.0 = สมบูรณ์, 0.5 = เท่ากับการสุ่ม)
#   สิ่งที่ผู้เรียนจะเห็น: ตารางผลของแต่ละเกณฑ์ และกราฟ ROC curve
# ============================================================================

# ผลลัพธ์ของทุกเกณฑ์ + ROC curve (โค้ดเตรียมไว้แล้ว)
# นำเข้าไลบรารี pandas (ใช้สร้างและแสดงตารางข้อมูล) ตั้งชื่อย่อว่า pd
import pandas as pd
# นำเข้าฟังก์ชัน 2 ตัวจาก scikit-learn
#   - roc_curve     : คำนวณจุดต่าง ๆ บนเส้นโค้ง ROC
#   - roc_auc_score : คำนวณพื้นที่ใต้เส้นโค้ง ROC (ROC AUC)
from sklearn.metrics import roc_curve, roc_auc_score
# ----------------------------------------------------------------------------
# ส่วนที่ 1: สร้างตารางผลลัพธ์ของเกณฑ์ตัดสินหลายค่า
# ----------------------------------------------------------------------------
# สร้าง list ว่างชื่อ rows สำหรับเก็บผลของแต่ละเกณฑ์ (1 เกณฑ์ = 1 แถวของตาราง)
rows = []
# วนลูปทีละเกณฑ์ t จาก list ของเกณฑ์ที่ต้องการเปรียบเทียบ
for t in [0.1, 0.3, 0.5, 0.7, 0.8, 0.9, 0.95]:
    # ตัดสินคำทำนายด้วยเกณฑ์ t: ความน่าจะเป็นตั้งแต่ t ขึ้นไป = 1 (ปอดอักเสบ) มิฉะนั้น 0 (ปกติ)
    p = (test_probs >= t).astype(int)
    # สร้าง confusion matrix (labels=[0, 1]) แล้วแตกเป็น 4 ตัวแปร TN, FP, FN, TP ด้วย .ravel()
    #   (ต่อท้ายชื่อด้วย _ เพื่อไม่ให้ทับตัวแปร tn, fp, fn, tp ของเซลล์ก่อนหน้า)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_true, p, labels=[0, 1]).ravel()
    # เพิ่มผลของเกณฑ์นี้ต่อท้าย rows ด้วย .append(...) (คำสั่งเดียวเขียนต่อ 2 บรรทัด)
    #   ข้อมูลอยู่ในรูปพจนานุกรม (dict) { "ชื่อคอลัมน์": ค่า, ... } ซึ่งจะกลายเป็น 1 แถวของตาราง
    #   - "เกณฑ์ (threshold)"        : t
    #   - "ตรวจไม่พบ (FN)"           : fn_
    #   - "ผลบวกลวง (FP)"            : fp_
    #   - "ความไว (sensitivity)"     : TP/(TP+FN) ปัดเป็นทศนิยม 3 ตำแหน่งด้วย round(ค่า, 3)
    #   - "ความจำเพาะ (specificity)" : TN/(TN+FP) ปัดเป็นทศนิยม 3 ตำแหน่ง
    rows.append({"เกณฑ์ (threshold)": t, "ตรวจไม่พบ (FN)": fn_, "ผลบวกลวง (FP)": fp_,
                 "ความไว (sensitivity)": round(tp_/(tp_+fn_), 3), "ความจำเพาะ (specificity)": round(tn_/(tn_+fp_), 3)})
# แปลง rows เป็นตาราง (DataFrame) ด้วย pd.DataFrame(rows) แล้วแสดงด้วย display(...)
#   - display : ฟังก์ชันของ Jupyter/Colab ที่แสดงตารางอย่างสวยงาม (ใช้ได้ทุกตำแหน่งในเซลล์ ต่างจากการแสดงอัตโนมัติที่ใช้ได้เฉพาะบรรทัดสุดท้าย)
display(pd.DataFrame(rows))

# ----------------------------------------------------------------------------
# ส่วนที่ 2: วาดเส้นโค้ง ROC
# ----------------------------------------------------------------------------
# คำนวณจุดบนเส้นโค้ง ROC ด้วย roc_curve(คำตอบจริง, ความน่าจะเป็น)
#   - fpr : อัตราผลบวกลวง (false positive rate) = 1 - ความจำเพาะ ของแต่ละเกณฑ์
#   - tpr : อัตราผลบวกจริง (true positive rate) = ความไว ของแต่ละเกณฑ์
#   - _   : ค่าที่ 3 คือรายการเกณฑ์ที่ใช้ ไม่ได้ใช้ต่อจึงตั้งชื่อเป็น _ (ธรรมเนียมแปลว่า "ไม่สนใจค่านี้")
fpr, tpr, _ = roc_curve(y_true, test_probs)
# คำนวณพื้นที่ใต้เส้นโค้ง ROC (AUC) จากคำตอบจริงและความน่าจะเป็น เก็บไว้ใน auc
auc = roc_auc_score(y_true, test_probs)
# สร้างรูปใหม่ขนาด 4×4 นิ้ว ด้วย plt.figure(figsize=(4, 4))
plt.figure(figsize=(4, 4))
# วาดเส้น 2 เส้น (2 คำสั่งคั่นด้วย ;)
#   1) plt.plot(fpr, tpr, label=...) : เส้นโค้ง ROC ของโมเดล (แกนนอน fpr, แกนตั้ง tpr)
#        - label : ชื่อเส้นในคำอธิบายกราฟ พร้อมค่า AUC ทศนิยม 3 ตำแหน่ง
#   2) plt.plot([0, 1], [0, 1], "k--", label=...) : เส้นทแยงจาก (0,0) ถึง (1,1) แทนการเดาสุ่ม (โยนเหรียญ)
#        - "k--" : รูปแบบเส้น k = สีดำ, -- = เส้นประ
plt.plot(fpr, tpr, label=f"ResNet-18 (AUC = {auc:.3f})"); plt.plot([0, 1], [0, 1], "k--", label="coin flip (AUC = 0.5)")
# ตกแต่งและแสดงกราฟ (5 คำสั่งคั่นด้วย ;)
#   - plt.xlabel(...) : ชื่อแกนนอน = 1 - ความจำเพาะ (อัตราการแจ้งเตือนผิด)
#   - plt.ylabel(...) : ชื่อแกนตั้ง = ความไว
#   - plt.legend()    : แสดงคำอธิบายเส้น (ใช้ข้อความจาก label)
#   - plt.title(...)  : หัวกราฟ
#   - plt.show()      : แสดงกราฟ
plt.xlabel("1 - specificity (false alarm rate)"); plt.ylabel("Sensitivity"); plt.legend(); plt.title("ROC curve"); plt.show()

💡 **คำอธิบายผลลัพธ์** ตารางแสดงความสัมพันธ์เชิงแลกเปลี่ยน (trade-off) ระหว่างความไวและความจำเพาะ โดยทั่วไปการเพิ่มค่าหนึ่งจะทำให้อีกค่าหนึ่งลดลง **ROC curve** แสดงความสัมพันธ์นี้สำหรับ *ทุก* เกณฑ์ตัดสิน และ **พื้นที่ใต้เส้นโค้ง (ROC AUC)** สรุปความสามารถของโมเดลในการแยกสองกลุ่มโดยไม่ขึ้นกับเกณฑ์ตัดสิน (1.0 = แยกได้สมบูรณ์, 0.5 = เทียบเท่าการสุ่ม) ROC AUC เป็นตัวชี้วัดที่ใช้บนตารางอันดับ 🏆 ด้วย

✍️ **คำถามที่ 3** หากนำโมเดลนี้ไปใช้ **คัดแยก (triage)** ภาพถ่ายรังสีทรวงอกของผู้ป่วยเด็ก (ระบุภาพที่ควรให้รังสีแพทย์อ่านก่อน) ผู้เรียนจะเลือกเกณฑ์ตัดสินใดจากตาราง เพราะเหตุใด และคำตอบจะเปลี่ยนไปหรือไม่ หากใช้โมเดลเพื่อ **ตัดภาวะปอดอักเสบออก (rule out)** โดยไม่มีแพทย์อ่านภาพ จงอธิบาย

✍️ **คำตอบ:**



## ส่วน B — การกำกับข้อมูลภาพและการแบ่งส่วนภาพ (แนวคิดและตัวอย่างโดยสังเขป)

การจำแนกภาพ (classification) ต้องการป้ายกำกับ *ระดับภาพ* ("ภาพนี้มีภาวะปอดอักเสบ") ขณะที่ **การตรวจหาวัตถุ (object detection)** และ **การแบ่งส่วนภาพ (segmentation)** ต้องการป้ายกำกับ *ระดับบริเวณ* ได้แก่ กรอบสี่เหลี่ยม หรือขอบเขตที่แม่นยำ (**mask**) รอบรอยโรค ซึ่งสร้างด้วยเครื่องมือกำกับข้อมูล (annotation tool) เช่น
- **[LabelMe](https://github.com/wkentaro/labelme)** — วาดรูปหลายเหลี่ยมหรือ mask ได้ มีขนาดเล็ก และใช้งานแบบออฟไลน์บนเครื่องของผู้ใช้
- **[CVAT](https://www.cvat.ai/)** — ใช้งานผ่านเว็บ รองรับผู้กำกับข้อมูลหลายคน ขั้นตอนการตรวจทาน และโครงการขนาดใหญ่

การวาด mask เป็นงานที่ใช้เวลามากและต้องอาศัยผู้เชี่ยวชาญ (มักเป็นรังสีแพทย์) ชุดข้อมูลสำหรับ segmentation จึงมักมีขนาดเล็กและมีต้นทุนสูง

เซลล์ด้านล่างแสดงตัวอย่าง segmentation แบบดั้งเดิม (Otsu threshold ซึ่งเลือกจุดตัดความสว่างโดยอัตโนมัติ) เพื่อแสดงลักษณะของ mask โดยไม่ต้องฝึกโมเดล การตรวจหาวัตถุด้วยโมเดลการเรียนรู้ของเครื่อง (machine learning) จะกล่าวถึงใน Lab 5 ผู้เรียนสามารถเปลี่ยนค่า `IMAGE_INDEX` (0–623) เพื่อพิจารณาภาพอื่น

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้
#   สาธิตแนวคิด "การแบ่งส่วนภาพ" (segmentation) อย่างง่ายด้วยวิธี Otsu threshold
#   วิธีนี้เลือกจุดตัดความสว่างให้อัตโนมัติ แล้วสร้าง "หน้ากาก" (mask) ระบุพิกเซลที่สว่างกว่าจุดตัด
#   (ในภาพถ่ายรังสีคือโครงสร้างที่ทึบรังสี เช่น กระดูก หัวใจ) โดยไม่ได้อาศัยความรู้ทางกายวิภาค
#   สิ่งที่ผู้เรียนจะเห็น: 3 ภาพเรียงกัน คือ ภาพต้นฉบับ, mask (ขาว = สว่างกว่าจุดตัด) และภาพซ้อนทับ
#   ผู้เรียนเปลี่ยน IMAGE_INDEX (0 ถึง 623) เพื่อดูภาพอื่นได้
# ============================================================================

# การแบ่งส่วนภาพด้วย Otsu threshold (โค้ดเตรียมไว้แล้ว)
# นำเข้าฟังก์ชัน threshold_otsu จากโมดูล skimage.filters ของไลบรารี scikit-image (ไลบรารีประมวลผลภาพ)
from skimage.filters import threshold_otsu
# เลือกลำดับภาพในชุดทดสอบที่จะใช้สาธิต (0 = ภาพแรก)
IMAGE_INDEX = 0

# ดึงภาพที่เลือกมาเตรียมเป็น NumPy array 2 มิติ (ทำตามลำดับ)
#   - test_ds[IMAGE_INDEX] : ข้อมูลรายการนั้น (ภาพ, ป้ายกำกับ)
#   - [0]                  : เลือกเฉพาะภาพ
#   - .squeeze()           : ตัดมิติขนาด 1 ออก ได้ภาพ 28×28
#   - .numpy()             : แปลงจากเทนเซอร์ของ PyTorch เป็น NumPy array ให้ scikit-image ใช้ได้
img = test_ds[IMAGE_INDEX][0].squeeze().numpy()
# คำนวณจุดตัดความสว่างอัตโนมัติด้วยวิธีของ Otsu
#   (เลือกค่าที่แบ่งพิกเซลเป็นกลุ่มมืดและกลุ่มสว่าง โดยให้ความแปรปรวนภายในแต่ละกลุ่มรวมกันน้อยที่สุด)
thr = threshold_otsu(img)
# สร้าง mask: เปรียบเทียบทุกพิกเซลกับจุดตัด ได้ True (สว่างกว่า) หรือ False (มืดกว่าหรือเท่ากับ)
mask = img > thr          # True = สว่างกว่าจุดตัด

# สร้างพื้นที่วาดภาพ 1 แถว 3 คอลัมน์ ขนาดกว้าง 10 นิ้ว สูง 3 นิ้ว (ax คือชุดช่องภาพ 3 ช่อง ax[0], ax[1], ax[2])
fig, ax = plt.subplots(1, 3, figsize=(10, 3))
# ช่องที่ 1 (ax[0]): แสดงภาพต้นฉบับโทนเทา และตั้งหัวภาพ (2 คำสั่งคั่นด้วย ;)
#   - หัวภาพแสดงลำดับภาพ และ 'pneumonia' หาก y_true[IMAGE_INDEX] เป็น 1 (ถือว่าเป็นจริง) มิฉะนั้น 'normal'
ax[0].imshow(img, cmap="gray"); ax[0].set_title(f"X-ray #{IMAGE_INDEX} ({'pneumonia' if y_true[IMAGE_INDEX] else 'normal'})")
# ช่องที่ 2 (ax[1]): แสดง mask (True = ขาว, False = ดำ) และหัวภาพที่ระบุจุดตัด ทศนิยม 2 ตำแหน่ง
ax[1].imshow(mask, cmap="gray"); ax[1].set_title(f"Otsu mask (thr={thr:.2f})")
# ช่องที่ 3 (ax[2]): แสดงภาพต้นฉบับ แล้ววาด mask ซ้อนทับ (3 คำสั่งคั่นด้วย ;)
#   - ax[2].imshow(img, cmap="gray")             : ภาพต้นฉบับเป็นพื้นหลัง
#   - ax[2].imshow(mask, alpha=0.3, cmap="Reds") : วาด mask ทับด้วยโทนสีแดง
#        - alpha=0.3   : ความทึบ 30% (โปร่งใส จึงยังเห็นภาพด้านล่าง)
#        - cmap="Reds" : บริเวณ True เป็นสีแดง
#   - ax[2].set_title("Overlay")                  : หัวภาพ "ภาพซ้อนทับ"
ax[2].imshow(img, cmap="gray"); ax[2].imshow(mask, alpha=0.3, cmap="Reds"); ax[2].set_title("Overlay")
# วนลูปแบบบรรทัดเดียว: สำหรับทุกช่องภาพ a ใน ax ให้ซ่อนแกนรอบภาพ
for a in ax: a.axis("off")
# แสดงภาพทั้งหมด
plt.show()

💡 **คำอธิบายผลลัพธ์** mask ระบุทุกพิกเซลที่สว่างกว่าจุดตัดซึ่งเลือกโดยอัตโนมัติ ในภาพถ่ายรังสีทรวงอก วิธีนี้จึงเลือกโครงสร้างที่มีความทึบรังสีเป็นหลัก (กระดูกสันหลัง หัวใจ เนื้อเยื่ออ่อน และกระดูก) ส่วนเนื้อปอดที่มีอากาศปรากฏเป็นสีดำจึง *ไม่* ถูกเลือก กฎอย่างง่ายเช่นนี้ไม่ได้อาศัยความรู้ทางกายวิภาค การแบ่งส่วนปอดหรือรอยโรคที่ถูกต้องจำเป็นต้องใช้โมเดลที่ฝึกด้วย mask ซึ่งผู้เชี่ยวชาญเป็นผู้วาด

✍️ **คำถามที่ 4** หากโรงพยาบาลต้องการระบบปัญญาประดิษฐ์ (AI) ที่ระบุขอบเขตบริเวณ consolidation บนภาพถ่ายรังสีทรวงอก โปรดระบุว่าต้องใช้ป้ายกำกับประเภทใด ผู้ใดควรเป็นผู้กำกับข้อมูล และจงอธิบายปัญหาที่อาจเกิดขึ้นหากผู้กำกับข้อมูลแต่ละคนให้ความเห็นไม่ตรงกัน

✍️ **คำตอบ:**



## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

**ส่วนนี้ไม่บังคับ และไม่นับเป็นส่วนของชิ้นงานที่ต้องส่ง** จัดไว้สำหรับผู้เรียนที่ต้องการพัฒนาโมเดลให้มีประสิทธิภาพสูงขึ้น

ให้ส่งค่า P(pneumonia) ของ **ภาพทดสอบทั้ง 624 ภาพ** (`id` = ลำดับที่ 0..623 ในชุด test ของ PneumoniaMNIST) **ตัวชี้วัด: ROC AUC** (ไม่ขึ้นกับเกณฑ์ตัดสิน จึงเป็นข้อมูลที่เสริมความไวและความจำเพาะที่คำนวณไว้แล้ว) หากต้องการส่งผล ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์ ⚙️ การตั้งค่าด้านบน แล้วรันเซลล์ดังกล่าวใหม่

> **หลักความซื่อสัตย์ทางวิชาการ (Honour code)** ป้ายกำกับที่ซ่อนไว้มาจากชุดข้อมูลสาธารณะ การค้นหาป้ายกำกับดังกล่าวจึงทำได้ง่ายแต่ไม่มีประโยชน์ต่อการเรียนรู้ ตารางอันดับมีไว้เพื่อสนับสนุนการเรียนรู้ และโน้ตบุ๊กของผู้เรียนจะได้รับการประเมินด้วย ห้ามแก้ไขผลการทำนายด้วยมือ

**Baseline** — เซลล์ด้านล่างส่งผลการทำนายของโมเดลจากส่วนหลักโดยไม่มีการปรับแก้ และสามารถใช้งานได้ทันที

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ไม่บังคับ — โจทย์ท้าทายบนตารางอันดับ)
#   สร้างไฟล์ผลลัพธ์ "baseline" (ผลอ้างอิงตั้งต้น) จากโมเดลที่ฝึกในส่วนหลัก
#     1) ให้โมเดลคำนวณความน่าจะเป็นของปอดอักเสบ P(pneumonia) ของภาพทดสอบทั้ง 624 ภาพ
#     2) บันทึกเป็นไฟล์ lab04_submission.csv (คอลัมน์ id และ prob)
#     3) หากกรอก NAME และ JOIN_CODE ไว้ในเซลล์การตั้งค่า จะส่งไฟล์ไปยังตารางอันดับ (ตัวชี้วัด: ROC AUC)
#   สิ่งที่ผู้เรียนจะเห็น: ข้อความยืนยันการบันทึกไฟล์ และคะแนน/อันดับ (หากส่งผล)
# ============================================================================

# 🏆 ส่งผล baseline: ความน่าจะเป็นจากโมเดลในส่วนหลักสำหรับภาพทดสอบ 624 ภาพ
# นำเข้าไลบรารี pandas (ตาราง/บันทึกไฟล์ CSV) ตั้งชื่อย่อว่า pd
import pandas as pd

# ----------------------------------------------------------------------------
# ฟังก์ชัน predict_probs: คำนวณความน่าจะเป็นของปอดอักเสบของทุกภาพในตัวป้อนข้อมูล แล้วคืนค่าเป็น list
#   - model  : โมเดลที่ใช้ทำนาย
#   - loader : ตัวป้อนข้อมูลของชุดที่ต้องการ
# ----------------------------------------------------------------------------
def predict_probs(model, loader):
    # ตั้งโมเดลเป็นโหมดประเมินผลด้วย model.eval() และสร้าง list ว่าง probs (2 คำสั่งคั่นด้วย ;)
    model.eval(); probs = []
    # ปิดการคำนวณ gradient ชั่วคราว เพราะทำนายอย่างเดียว ไม่ได้ฝึก
    with torch.no_grad():
        # วนลูปดึงภาพ x ทีละชุดย่อย ส่วนป้ายกำกับไม่ได้ใช้จึงตั้งชื่อเป็น _ (ไม่สนใจค่านี้)
        #   ตัวป้อนข้อมูลไม่สลับลำดับ ผลจึงเรียงตรงกับลำดับภาพทดสอบ
        for x, _ in loader:                        # shuffle=False -> ลำดับ = ดัชนีภาพทดสอบ 0..623
            # คำนวณความน่าจะเป็นของปอดอักเสบ แล้วต่อท้ายเข้า probs
            #   x.to(device) -> model(...) ได้คะแนน 2 ค่า -> torch.softmax(..., dim=1) แปลงเป็นความน่าจะเป็น
            #   -> [:, 1] เลือกคอลัมน์ "ปอดอักเสบ" -> .cpu() ย้ายมา CPU -> .tolist() แปลงเป็น list
            probs += torch.softmax(model(x.to(device)), dim=1)[:, 1].cpu().tolist()
    # ส่ง list ความน่าจะเป็นกลับไป
    return probs

# เรียกฟังก์ชันกับโมเดลหลัก net และตัวป้อนข้อมูลชุดทดสอบ เก็บผลไว้ใน probs
probs = predict_probs(net, test_loader)
# assert ตรวจเงื่อนไข: ต้องมีผลครบ 624 ค่า หากไม่ครบจะหยุดและแสดงข้อความเตือนหลังจุลภาค
assert len(probs) == 624, "ต้องมีผลการทำนายครบ 624 ภาพ"
# สร้างตารางแล้วบันทึกเป็นไฟล์ CSV
#   - pd.DataFrame({...})          : สร้างตารางจาก dict โดยแต่ละกุญแจคือชื่อคอลัมน์
#        - "id": range(len(probs)) : ลำดับภาพ 0, 1, ..., 623
#        - "prob": probs           : ความน่าจะเป็นของปอดอักเสบของแต่ละภาพ
#   - .to_csv("lab04_submission.csv", ...) : บันทึกตารางเป็นไฟล์ CSV ชื่อนี้
#        - index=False             : ไม่บันทึกคอลัมน์เลขแถวอัตโนมัติของ pandas
pd.DataFrame({"id": range(len(probs)), "prob": probs}).to_csv("lab04_submission.csv", index=False)
# แจ้งว่าบันทึกไฟล์เรียบร้อย
print("บันทึกไฟล์ lab04_submission.csv เรียบร้อยแล้ว")
# ตรวจว่ากรอกทั้ง NAME และ JOIN_CODE ในเซลล์การตั้งค่าแล้วหรือไม่ (ข้อความว่างถือเป็นเท็จ)
if NAME and JOIN_CODE:
    # ส่งไฟล์ไปยังตารางอันดับด้วยฟังก์ชัน submit_to_leaderboard ที่สร้างไว้ในเซลล์การตั้งค่า
    submit_to_leaderboard("lab04_submission.csv")
# else : กรณีที่เงื่อนไข if เป็นเท็จ (ยังไม่ได้กรอก)
else:
    # แจ้งว่ายังไม่ได้ส่งผล เพราะยังไม่ได้ระบุ NAME / JOIN_CODE (การส่งไม่บังคับ)
    print("ยังไม่ได้ระบุ NAME / JOIN_CODE ในเซลล์การตั้งค่า จึงไม่ได้ส่งผลไปยังตารางอันดับ (ไม่บังคับ)")

### แนวทางการพัฒนาให้ดีกว่า baseline
- เพิ่มจำนวน epoch (เช่น 10–20) และลดอัตราการเรียนรู้ (learning rate) เช่น `lr=3e-4`
- การเพิ่มข้อมูล (data augmentation) ในชุดฝึก เช่น `transforms.RandomRotation(10)`, `transforms.RandomAffine(0, translate=(0.05, 0.05))`
- ใช้ภาพที่มีขนาดใหญ่ขึ้น: `size=64` หรือ `size=128` (MedMNIST+) ซึ่งมีรายละเอียดมากขึ้นแต่ใช้เวลาประมวลผลนานขึ้น
- จัดการความไม่สมดุลของคลาส: `nn.CrossEntropyLoss(weight=...)` หรือ weighted sampler
- เก็บ checkpoint ที่ให้ AUC บนชุด **validation** สูงที่สุด (ห้ามปรับแต่งโมเดลโดยอาศัยชุดทดสอบ)
- ใช้โครงข่ายขนาดใหญ่ขึ้น (ResNet-34/50) หรือ test-time augmentation (เฉลี่ยผลการทำนายจากภาพที่พลิกหรือเลื่อนตำแหน่ง)
- สำหรับการศึกษาเพิ่มเติม: เปลี่ยนเป็น `BreastMNIST` (ภาพอัลตราซาวด์) หรือ `DermaMNIST` (รอยโรคผิวหนัง) โดยแก้ไขเพียงบรรทัดเดียว แล้วเปรียบเทียบระดับความยาก (ไม่ใช้กับตารางอันดับ)

เซลล์ (ไม่บังคับ) ด้านล่างเป็นจุดเริ่มต้น (ภาพ 64×64 + data augmentation + จำนวน epoch มากขึ้น ใช้เวลาประมาณ 5 นาทีบน T4 GPU และนานกว่ามากบน CPU) ผู้เรียนสามารถแก้ไขได้ตามต้องการ

In [ ]:
# ============================================================================
# 🎯 วัตถุประสงค์ของเซลล์นี้ (ไม่บังคับ — โจทย์ท้าทายบนตารางอันดับ)
#   ตัวอย่างการพัฒนาโมเดลให้ดีกว่า baseline โดย
#     - ใช้ภาพขนาดใหญ่ขึ้น (64×64 พิกเซล) ซึ่งมีรายละเอียดมากขึ้น
#     - การเพิ่มข้อมูล (data augmentation): หมุนและเลื่อนภาพฝึกแบบสุ่มเล็กน้อย ให้โมเดลทนต่อความแตกต่างของการจัดท่า
#     - ฝึกนานขึ้น (8 รอบ) ด้วยอัตราการเรียนรู้ (learning rate) ที่ต่ำลง
#   จากนั้นเปรียบเทียบ ROC AUC บนชุดตรวจสอบ (validation) ระหว่าง baseline กับโมเดลใหม่
#   แล้วบันทึกไฟล์ผลลัพธ์ใหม่ (และส่งตารางอันดับหากกรอก NAME / JOIN_CODE)
#   ผู้เรียนแก้ไขค่าต่าง ๆ ได้ตามต้องการ (ใช้เวลาฝึกนานกว่าส่วนหลักพอสมควร)
# ============================================================================

# 🏆 (ไม่บังคับ) โมเดลที่ปรับปรุงแล้ว — สามารถแก้ไขได้ตามต้องการ แล้วรันใหม่
# นำเข้าฟังก์ชันคำนวณ ROC AUC จาก scikit-learn
from sklearn.metrics import roc_auc_score
# กำหนดค่า 3 ตัวพร้อมกัน
#   - SIZE = 64         : ขนาดภาพ 64×64 พิกเซล
#   - EPOCHS_PLUS = 8   : จำนวนรอบการฝึก 8 รอบ
#   - LR = 3e-4         : อัตราการเรียนรู้ 0.0003 (3×10⁻⁴)
SIZE, EPOCHS_PLUS, LR = 64, 8, 3e-4

# ----------------------------------------------------------------------------
# เตรียมข้อมูล: ขั้นตอนแปลงภาพแบบมีการเพิ่มข้อมูล (ใช้กับชุดฝึกเท่านั้น)
# ----------------------------------------------------------------------------
# สร้างขั้นตอนแปลงภาพสำหรับชุดฝึก aug_tf ด้วย transforms.Compose([...]) (ทำตามลำดับ)
aug_tf = transforms.Compose([
    # transforms.RandomAffine(...) : ปรับภาพแบบสุ่มทุกครั้งที่ดึงภาพ
    #   - degrees=10               : หมุนภาพแบบสุ่มไม่เกิน ±10 องศา
    #   - translate=(0.05, 0.05)   : เลื่อนภาพแบบสุ่มไม่เกิน 5% ของความกว้าง (แนวนอน) และความสูง (แนวตั้ง)
    transforms.RandomAffine(degrees=10, translate=(0.05, 0.05)),
    # แปลงเป็นเทนเซอร์ช่วง 0-1 (ToTensor) แล้วปรับเป็นช่วง -1 ถึง 1 (Normalize ค่าเฉลี่ย 0.5 ส่วนเบี่ยงเบน 0.5)
    transforms.ToTensor(), transforms.Normalize([.5], [.5]),
])
# สร้างตัวป้อนข้อมูลชุดฝึก (คำสั่งเดียวซ้อนกัน 2 ชั้น)
#   - PneumoniaMNIST(split="train", transform=aug_tf, download=True, size=SIZE) : ชุดฝึก ภาพ 64×64 พร้อมการเพิ่มข้อมูล
#   - DataLoader(..., batch_size=64, shuffle=True) : ป้อนครั้งละ 64 ภาพ สลับลำดับแบบสุ่ม
train_plus = DataLoader(PneumoniaMNIST(split="train", transform=aug_tf, download=True, size=SIZE), batch_size=64, shuffle=True)
# สร้างตัวป้อนข้อมูลชุดตรวจสอบ ภาพ 64×64 ใช้การแปลงปกติ tf (ไม่สุ่มดัดแปลงภาพ) ครั้งละ 128 ภาพ ไม่สลับลำดับ
val_plus   = DataLoader(PneumoniaMNIST(split="val",   transform=tf, download=True, size=SIZE), batch_size=128)
# สร้างตัวป้อนข้อมูลชุดทดสอบ ภาพ 64×64 ใช้การแปลงปกติ tf ครั้งละ 128 ภาพ ไม่สลับลำดับ (ลำดับ 0..623 ตรงกับ id)
test_plus  = DataLoader(PneumoniaMNIST(split="test",  transform=tf, download=True, size=SIZE), batch_size=128)

# ----------------------------------------------------------------------------
# สร้างและฝึกโมเดลใหม่ (net2) ด้วยวิธีเดียวกับแบบฝึกหัดที่ 1
# ----------------------------------------------------------------------------
# สร้าง ResNet-18 ใหม่พร้อมค่าน้ำหนักที่ฝึกกับ ImageNet แล้ว
net2 = models.resnet18(weights="IMAGENET1K_V1")
# รวมตัวกรองของชั้นแรกจาก 3 ช่องสีเป็น 1 ช่อง (dim=1 = มิติช่องสี, keepdim=True = คงมิติไว้) เก็บไว้ใน w
w = net2.conv1.weight.data.sum(dim=1, keepdim=True)
# 2 คำสั่งคั่นด้วย ;
#   1) แทนที่ชั้นแรกด้วย nn.Conv2d ที่รับภาพ 1 ช่อง ส่งออก 64 ตัวกรอง
#        (kernel_size=7 ตัวกรอง 7×7, stride=2 เลื่อนครั้งละ 2, padding=3 เติมขอบ 3, bias=False ไม่ใช้ bias)
#   2) ใส่ค่าน้ำหนัก w ที่เตรียมไว้ให้ชั้นใหม่ เพื่อคงความรู้เดิมจาก ImageNet
net2.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False); net2.conv1.weight.data = w
# แทนที่ชั้นสุดท้ายด้วย nn.Linear ที่รับ 512 ค่า (in_features เดิม) และส่งออก 2 ค่า (ปกติ / ปอดอักเสบ)
net2.fc = nn.Linear(net2.fc.in_features, 2)
# ฝึกโมเดลด้วยฟังก์ชัน train_model จากเซลล์การฝึก แล้วเก็บโมเดลที่ฝึกแล้วไว้ใน net2
#   - net2.to(device) : ย้ายโมเดลไปยัง GPU/CPU ก่อนฝึก
#   - train_plus      : ตัวป้อนข้อมูลชุดฝึก (พร้อมการเพิ่มข้อมูล)
#   - val_plus        : ตัวป้อนข้อมูลชุดตรวจสอบ
#   - EPOCHS_PLUS     : จำนวนรอบ (8)
#   - lr=LR           : อัตราการเรียนรู้ 0.0003
net2 = train_model(net2.to(device), train_plus, val_plus, EPOCHS_PLUS, lr=LR)

# ----------------------------------------------------------------------------
# เปรียบเทียบผลบนชุดตรวจสอบ (ไม่ใช้ชุดทดสอบในการตัดสินใจ เพื่อป้องกันการปรับโมเดลให้เข้ากับชุดทดสอบ)
# ----------------------------------------------------------------------------
# ดึงคำตอบจริงของชุดตรวจสอบ
#   - val_plus.dataset : ชุดข้อมูลที่อยู่ในตัวป้อนข้อมูล
#   - .labels          : ป้ายกำกับทั้งหมด
#   - .ravel()         : เรียงให้เป็นแถวเดียว
val_y = val_plus.dataset.labels.ravel()
# แสดง ROC AUC บนชุดตรวจสอบของทั้งสองโมเดล (คำสั่งเดียวเขียนต่อ 2 บรรทัด ทศนิยม 3 ตำแหน่ง)
#   - roc_auc_score(val_y, predict_probs(net, val_loader)) : AUC ของ baseline (ภาพ 28×28)
#   - roc_auc_score(val_y, predict_probs(net2, val_plus))  : AUC ของโมเดลที่ปรับปรุง (ภาพ 64×64)
#   (ทั้งสองตัวป้อนข้อมูลไม่สลับลำดับ ผลทำนายจึงเรียงตรงกับ val_y)
print(f"AUC บนชุด validation: baseline={roc_auc_score(val_y, predict_probs(net, val_loader)):.3f}  "
      f"โมเดลที่ปรับปรุง={roc_auc_score(val_y, predict_probs(net2, val_plus)):.3f}")

# ----------------------------------------------------------------------------
# สร้างไฟล์ผลลัพธ์ใหม่จากโมเดลที่ปรับปรุง และส่งตารางอันดับ (หากกรอกข้อมูลไว้)
# ----------------------------------------------------------------------------
# คำนวณความน่าจะเป็นของปอดอักเสบของภาพทดสอบทั้งหมด ด้วยโมเดล net2
probs2 = predict_probs(net2, test_plus)
# ตรวจว่ามีผลครบ 624 ภาพ หากไม่ครบจะหยุดพร้อมข้อความเตือน
assert len(probs2) == 624, "ต้องมีผลการทำนายครบ 624 ภาพ"
# สร้างตาราง (คอลัมน์ id = ลำดับภาพ, prob = ความน่าจะเป็น) แล้วบันทึกทับไฟล์ CSV เดิม (index=False ไม่บันทึกเลขแถว)
pd.DataFrame({"id": range(len(probs2)), "prob": probs2}).to_csv("lab04_submission.csv", index=False)
# แจ้งว่าบันทึกไฟล์ของโมเดลที่ปรับปรุงเรียบร้อย
print("บันทึกไฟล์ lab04_submission.csv เรียบร้อยแล้ว (โมเดลที่ปรับปรุง)")
# หากกรอกทั้ง NAME และ JOIN_CODE ไว้แล้ว ให้ส่งผล
if NAME and JOIN_CODE:
    # ส่งไฟล์ไปยังตารางอันดับ แล้วแสดงคะแนนและอันดับ
    submit_to_leaderboard("lab04_submission.csv")

## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

1. โน้ตบุ๊กนี้ที่ **รันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (แสดง confusion matrix ความไวและความจำเพาะ ภาพที่โมเดลจำแนกผิด และตารางเกณฑ์ตัดสิน)
2. คำตอบของ ✍️ **คำถามที่ 1–4** ในเซลล์คำตอบ
3. *(ไม่บังคับ)* ผู้เรียนที่เข้าร่วมโจทย์ท้าทาย 🏆 สามารถระบุชื่อบนตารางอันดับและคะแนนสูงสุดไว้ด้านล่าง

ให้ส่งงานตามช่องทางที่รายวิชากำหนด หากมีข้อสงสัย สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom

✍️ **คำตอบ:** *(ไม่บังคับ: ชื่อบนตารางอันดับ / คะแนน)*

